# DATE25（對照組）在 Attar 上：superblue18

論文：Li, Yang, Ai, Zhu，〈Analytically-Derived Hybrid Net–Pin Weighting for Timing-Driven Global Placement〉，DAC'26。

這個 notebook 跑 P11 的對照組 **DATE25-TDP** 的 pin 對加權（移植到 [Attar](https://github.com/RayChang987/Attar)，
GPU 上的 placement 與 STA 函式庫），不需要編譯 DREAMPlace。流程：global placement（`src/gp_tdp.py`，每 15 次迭代用
timing 更新 pin 對權重）→ Abacus 合法化 → Attar 的 STA 量測。

P11 的方法在另一個 notebook：`p11_attar_superblue18.ipynb`。這裡跑出的 TNS / WNS 可以填到那邊算改善倍數。

**看數字時請注意**
1. Attar 的 timer 和 OpenSTA 一致（superblue 上每個 endpoint 誤差約 1 ps）。論文用的是 DATE25-TDP 內建的 OpenTimer，
   它對部分 I/O 路徑的處理不同，所以這裡的 TNS 絕對值比較大，**不能和論文 Table 1 直接比較**。
2. 這裡的 DATE25 是我們移植到 Attar 的版本，比官方 DATE25-TDP 弱約 1.45 倍。

可以和論文直接比較的結果，是在論文平台（DREAMPlace + OpenTimer）上跑 8 個設計：P11 比 DATE25 好 TNS 1.52 倍、
WNS 1.09 倍，論文是 1.62 / 1.19。那需要編譯 DATE25-TDP，不在這些 notebook 裡。

**執行環境**：請使用 GPU runtime（執行階段 → 變更執行階段類型 → GPU）。第一次執行會下載設計檔（約 60 MB），
並編譯 Triton kernel（幾分鐘，只有第一次）。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), '請切換到 GPU runtime：執行階段 -> 變更執行階段類型 -> GPU'

In [ ]:
!pip install -q attar==1.0.4

## 原始碼
這個示範需要的程式碼（12 個檔案，寫到 `/content/p11/src`）。直接執行即可，不用細看。

In [ ]:
import os
os.makedirs('/content/p11/src', exist_ok=True)
%cd /content/p11

In [ ]:
%%writefile src/__init__.py
"""P11: analytically-derived hybrid net-pin weighting for timing-driven global placement (DAC 2026).

Paper: Li, Yang, Ai, Zhu, "Analytically-Derived Hybrid Net-Pin Weighting for Timing-Driven Global Placement", DAC '26.
Built on Attar 1.0.4: attar.naive.global_place's `extra` / `callback` hooks and TimingResult.critical_paths.
"""

In [ ]:
%%writefile src/run.py
#!/usr/bin/env python3
"""Batch driver: python -m src.run --methods W0,W2,ours --designs superblue18 superblue16 --seeds 0

One run per process (the CPU path has no CUDA memory to leak, but the timer, the weight store and the GP state are
per design anyway). Results land in results/<method>/<design>/seed<k>/.
"""
import argparse
import json
import os
import sys
import time

from . import common, tdgp


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument('--methods', default='ours', help='comma separated, see src/tdgp.py METHODS')
    ap.add_argument('--designs', default='superblue18', help='comma separated: superblueN or a full zoo name')
    ap.add_argument('--seeds', default='0', help='comma separated')
    ap.add_argument('--max-iters', type=int, default=2500)
    ap.add_argument('--stop-overflow', type=float, default=0.07)
    ap.add_argument('--start', type=int, default=500, help='first iteration that times the placement')
    ap.add_argument('--m', type=int, default=20, help='iterations between weight updates')
    ap.add_argument('--beta', default='2.5e-5', help="the paper's 2.5e-5, or 'auto' (gradient balance)")
    ap.add_argument('--beta-target', type=float, default=0.3, help='||grad PP|| / ||grad WL|| for --beta auto')
    ap.add_argument('--t1', type=float, default=3e5)
    ap.add_argument('--t2', type=float, default=1e4)
    ap.add_argument('--net-bump', type=float, default=None, help='calibrate the net factor: 95th pct multiplier = 1+bump')
    ap.add_argument('--lin-scale', type=float, default=1.0)
    ap.add_argument('--quad-scale', type=float, default=1.0)
    ap.add_argument('--net-scale', type=float, default=1.0, help='scale the net-level weights (unit calibration)')
    ap.add_argument('--net-cap', type=float, default=None, help='clip the net-weight multiplier (report it)')
    ap.add_argument('--start-ov', type=float, default=None,
                    help='start timing at the first iteration with density overflow <= this (overrides --start)')
    ap.add_argument('--beta-units', default='um', choices=('um', 'site'),
                    help="'site': --beta is in the paper's DREAMPlace units and is converted (tdgp.paper_beta)")
    ap.add_argument('--beta-target-from', default=None,
                    help="with --beta auto: follow another run's measured timing force (its trace.json r_time)")
    ap.add_argument('--beta-target-scale', type=float, default=1.0, help='multiply the --beta auto target')
    ap.add_argument('--beta-fixed', action='store_true',
                    help='--beta auto calibrates once at the first update and keeps beta (the paper: constant beta)')
    ap.add_argument('--gp', default='tdp', choices=('tdp', 'attar'),
                    help="tdp: src/gp_tdp.py (live net weights, DREAMPlace's net-weighted preconditioner); attar: naive")
    ap.add_argument('--m-d25', type=int, default=None, help='update period of the DATE25 scheme (official: 15)')
    ap.add_argument('--target-density', type=float, default=None)
    ap.add_argument('--bins', type=int, default=None)
    ap.add_argument('--trace-every', type=int, default=None, help='trace W0 every k iterations after the start')
    ap.add_argument('--device', default=None)
    ap.add_argument('--root', default=None, help='results root (default results/)')
    ap.add_argument('--no-lg', action='store_true', help='skip legalization (debug only)')
    ap.add_argument('--quiet', action='store_true')
    a = ap.parse_args(argv)
    frac = os.environ.get('P11_GPU_FRACTION')       # shared GPU: a hard cap on this process's memory
    if frac:
        import torch
        if torch.cuda.is_available():
            torch.cuda.set_per_process_memory_fraction(float(frac), 0)
    beta = a.beta if a.beta == 'auto' else float(a.beta)
    gp_kw = {k: v for k, v in (('target_density', a.target_density), ('bins', a.bins)) if v is not None}

    methods = [x for x in a.methods.split(',') if x]
    designs = [x for x in a.designs.split(',') if x]
    seeds = [int(x) for x in a.seeds.split(',') if x]
    if len(methods) * len(designs) * len(seeds) > 1 and common.DEVICE != 'cpu' and a.device != 'cpu':
        # GPU: one process per run (the GP's CUDA graphs and the timer keep ~10 GB that a 12 GB card cannot hold twice)
        import subprocess
        argv = list(sys.argv[1:] if argv is None else argv)

        def drop(xs, flag):
            out_, skip = [], False
            for x in xs:
                if skip:
                    skip = False
                    continue
                if x == flag:
                    skip = True
                    continue
                if x.startswith(flag + '='):
                    continue
                out_.append(x)
            return out_
        for f in ('--methods', '--designs', '--seeds'):
            argv = drop(argv, f)
        for design in designs:
            for method in methods:
                for seed in seeds:
                    subprocess.run([sys.executable, '-m', 'src.run', '--methods', method, '--designs', design,
                                    '--seeds', str(seed)] + argv, check=False)
        return 0
    out = []
    for design in designs:
        for method in methods:
            for seed in seeds:
                t0 = time.time()
                print('=== %s %s seed %d ===' % (method, design, seed), flush=True)
                try:
                    metrics, trace, store = tdgp.run(design, method=method, seed=seed, dev=a.device, root=a.root,
                                                     max_iters=a.max_iters, start=a.start, beta=beta,
                                                     m=(a.m_d25 if a.m_d25 and method.startswith('D25') else a.m),
                                                     start_ov=a.start_ov, beta_units=a.beta_units, gp_kw=gp_kw,
                                                     trace_every=a.trace_every,
                                                     beta_target_from=a.beta_target_from,
                                                     beta_target_scale=a.beta_target_scale,
                                                     beta_fixed=a.beta_fixed, gp_engine=a.gp,
                                                     t1=a.t1, t2=a.t2, net_cap=a.net_cap, lg=not a.no_lg, net_scale=a.net_scale,
                                                     net_bump=a.net_bump, lin_scale=a.lin_scale,
                                                     quad_scale=a.quad_scale,
                                                     verbose=not a.quiet, stop_overflow=a.stop_overflow,
                                                     beta_target=a.beta_target)
                except Exception as exc:                                   # one failure must not stop the batch
                    import traceback
                    traceback.print_exc()
                    out.append({'design': design, 'method': method, 'seed': seed, 'error': repr(exc)})
                    continue
                out.append(metrics)
                pre = metrics.get('pre_lg', {})
                print('--- %s %s seed %d: legal %s  HPWL %.5g  WNS %.4g  TNS %.4g  | pre-LG HPWL %.5g WNS %.4g '
                      'TNS %.4g  (%.0f s)'
                      % (method, design, seed, metrics.get('legal'), metrics.get('hpwl_um', float('nan')),
                         metrics.get('wns', float('nan')), metrics.get('tns', float('nan')),
                         pre.get('hpwl_um', float('nan')), pre.get('wns', float('nan')), pre.get('tns', float('nan')),
                         time.time() - t0),
                      flush=True)
    if len(out) > 1:
        path = os.path.join(a.root or common.RESULTS, 'batch_%s.json' % time.strftime('%m%d_%H%M%S'))
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, 'w') as f:
            json.dump(out, f, indent=1)
        print('batch summary ->', path)
    return 0


if __name__ == '__main__':
    sys.exit(main())

In [ ]:
%%writefile src/common.py
"""Shared harness for the P11 reproduction: loading, the one ruler every method is measured with, results I/O.

The ruler (VALIDATION.md): legality, setup timing (Attar's OpenSTA-aligned timer on placement parasitics, i.e.
estimate_parasitics -placement) and HPWL, always after legalization and always with the netlist's own net weights
restored. DREAMPlace 4.0 / OpenROAD read-backs are measured through the same function.
"""
import json
import os
import time

import torch

import attar
from attar import Evaluator, metrics as M

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RESULTS = os.path.join(ROOT, 'results')
DEVICES = ('cpu',)
DESIGNS = ['superblue18', 'superblue16', 'superblue4', 'superblue5', 'superblue1', 'superblue3', 'superblue7',
           'superblue10']
DEVICE = os.environ.get('P11_DEVICE', 'cpu')


def zname(design):
    """The zoo name of an ICCAD 2015 superblue design."""
    return design if '/' in design else 'after_fp/iccad2015/%s' % design


def load(design, dev=None, unplace=True):
    d = attar.load(zname(design), device=dev or DEVICE)
    if unplace:
        d.unplace()
    return d


def hpwl_unweighted(d, pos=None):
    """HPWL with the netlist's own net weights (d.n_weight is scaled while timing-driven GP runs)."""
    p = d.pin_pos(d.pos if pos is None else pos)
    lo, hi = M.net_bbox(p, d.p_net, len(d.n_weight))
    return float((hi - lo).sum())


def measure(d, timing=True, legality=True):
    """The ruler. Call after legalization, with d.n_weight restored."""
    out = {}
    if legality:
        leg = Evaluator(d).legality()
        out['legal'] = bool(leg['legal'])
    out['hpwl_um'] = hpwl_unweighted(d)
    if timing:
        r = d.timing()
        out.update(wns=float(r.wns), tns=float(r.tns), setup_violations=int(r.violations),
                   endpoints=int(len(r.endpoints)))
    return out


def run_dir(method, design, seed, root=None):
    return os.path.join(root or RESULTS, method, os.path.basename(design).replace('/', '_'), 'seed%d' % seed)


def save_run(path, config, metrics, trace=None, pos=None, gp_pos=None):
    os.makedirs(path, exist_ok=True)
    if gp_pos is not None:                     # the global placement before legalization (re-legalize / re-measure)
        torch.save(gp_pos.detach().cpu(), os.path.join(path, 'gp_pos.pt'))
    with open(os.path.join(path, 'config.json'), 'w') as f:
        json.dump(config, f, indent=1)
    with open(os.path.join(path, 'metrics.json'), 'w') as f:
        json.dump(metrics, f, indent=1)
    if trace is not None:
        with open(os.path.join(path, 'trace.json'), 'w') as f:
            json.dump(trace, f)
    if pos is not None:
        torch.save(pos.detach().cpu(), os.path.join(path, 'pos.pt'))


def gamma_base(d):
    """DREAMPlace's WA gamma base as Attar's global_place computes it: 2 (bin width + bin height)."""
    from attar.kernels import default_bins
    n = int(d.movable.sum())
    nb = default_bins(n)
    cx0, cy0, cx1, cy1 = [float(v) for v in d.core_box[0]]
    return 2.0 * ((cx1 - cx0) / nb + (cy1 - cy0) / nb)


def gamma_of(ov, base):
    """The WA gamma global_place uses at this density overflow (naive/gp.py)."""
    ov = min(max(float(ov), 0.0), 1.0)
    return base * 10 ** ((ov - 0.1) * 20 / 9 - 1)


def stamp():
    return time.strftime('%Y-%m-%d %H:%M:%S')

In [ ]:
%%writefile src/tdgp.py
"""The P11 flow: wirelength-driven GP, then the hybrid net-pin timing objective (paper eq. 19), then legalization.

From iteration `start` (500 in the paper) every `m` = 20 iterations: time the unlegalized placement, walk the worst
paths of every violating endpoint, accumulate the six weights, scale d.n_weight in place by eq. 19's net-level factor,
and keep going with PP1 + PP2 added through global_place's `extra` hook.

Methods (same code, the switches decide what enters the objective):
  W0    wirelength only                                         (Attar naive GP + LG)
  W1    OpenROAD gpl's timing-driven net weighting, two passes   (Attar's examples/timing_driven.py)
  W2    quadratic pin-pair term alone                             (DATE25-TDP's core, our proxy)
  ours  net + linear + quadratic                                  (the paper)
  A1    linear + quadratic (no net)      A2  net + quadratic (no linear)
  A3    net + linear (no quadratic)      A4  ours without the b_i*c term in the net weight
"""
import json
import os
import time

import torch

import attar
from attar import Evaluator, naive
from attar import metrics as M

from . import common, gp_tdp, gpov, objective, weights

METHODS = {
    'W0': dict(use_net=False, lin=False, quad=False, skip_b=False),
    'W1': dict(use_net=False, lin=False, quad=False, skip_b=False),
    'W2': dict(use_net=False, lin=False, quad=True, skip_b=False),
    'ours': dict(use_net=True, lin=True, quad=True, skip_b=False),
    # ours with the timing force split evenly between the net term and the pin-pair terms (the paper gives no units,
    # so the net / pin ratio of eqs. 17-18 is not fixed by it; 'ours' keeps the ratio of r, c in ps, fF, um)
    'oursB': dict(use_net=True, lin=True, quad=True, skip_b=False, net_share=0.5),
    'A1': dict(use_net=False, lin=True, quad=True, skip_b=False),
    'A2': dict(use_net=True, lin=False, quad=True, skip_b=False),
    'A3': dict(use_net=True, lin=True, quad=False, skip_b=False),
    'A4': dict(use_net=True, lin=True, quad=True, skip_b=True),
    'N': dict(use_net=True, lin=False, quad=False, skip_b=False),      # net-level term alone (all the force)
    'P': dict(use_net=False, lin=True, quad=False, skip_b=False),      # linear pin pairs alone
    'D25': dict(use_net=False, lin=False, quad=True, skip_b=False, scheme='date25'),   # DATE25-TDP's own scheme
    # DATE25's pairs at their official beta, unchanged, + the derived net term whose force = net_ratio x the pair
    # term's force at each update (the protocol of src/p11_dreamplace.py add_net on DATE25-TDP itself)
    'D25N': dict(use_net=True, lin=False, quad=True, skip_b=False, scheme='date25', net_ratio=1.0),
    'D25N025': dict(use_net=True, lin=False, quad=True, skip_b=False, scheme='date25', net_ratio=0.25),   # diagnostics
    'D25N05': dict(use_net=True, lin=False, quad=True, skip_b=False, scheme='date25', net_ratio=0.5),
    'D25N2': dict(use_net=True, lin=False, quad=True, skip_b=False, scheme='date25', net_ratio=2.0),
    # D25Nns: the same, but the net term at the paper's printed beta (2.5e-5, DREAMPlace site units) with the derived
    # weights read in ns instead of ps -- no force matching; cumulative weights ramp the force up (report_v3 sec. 5)
    'D25Nns': dict(use_net=True, lin=False, quad=True, skip_b=False, scheme='date25', net_literal=1e-3),
    'D25H': dict(use_net=True, lin=False, quad=True, skip_b=False, scheme='date25', net_share=0.5),  # DATE25 pairs
    # + the derived net term, the force split evenly (oursB with DATE25's pair weights)
    'D25P': dict(use_net=False, lin=True, quad=True, skip_b=False, scheme='date25'),  # DATE25 pairs + linear pin pairs
}


def _ev(d):
    """One Evaluator per design (each one allocates its own density grid: a new one every few iterations ran a
    12 GB card out of memory next to the placer)."""
    ev = getattr(d, '_p11_ev', None)
    if ev is None:
        ev = Evaluator(d)
        d._p11_ev = ev
    return ev


def wa_grad(d, pos, w, ov, gamma_base):
    """The WA wirelength gradient on the cells with net weights w."""
    ev = _ev(d)
    p = d.pin_pos(pos)
    _wl, gp = M.wa_value_grad(p, ev.net, ev.E, common.gamma_of(ov, gamma_base), w)
    return torch.zeros_like(pos).index_add_(0, d.p_cell, gp)


def timing_grads(d, store, state, pos, t1, t2, cfg, base, lin_scale=1.0, quad_scale=1.0, net_scale=1.0, net_t=None):
    """At beta = 1: the pin-pair terms' gradient, the net term's gradient (the WA gradient with the weights the net
    factor adds, base * net_scale * (t1 w_tns + t2 w_wns)) and the plain wirelength's gradient (base weights).
    Both timing parts are linear in beta, so beta * (g_pp + g_net) is the whole timing force at a given beta.
    net_t: the (t1, t2) of the net term when the pairs use others (DATE25's scheme: (1, 0))."""
    nt1, nt2 = net_t or (t1, t2)
    _v, g_pp = objective.value_at(d, store, pos, t1, t2, cfg['lin'], cfg['quad'], state['ov'], state['gamma_base'],
                                  lin_scale, quad_scale)
    g_net = torch.zeros_like(pos)
    if cfg['use_net']:
        wn = store.net_tns_nob if cfg['skip_b'] else store.net_tns
        g_net = wa_grad(d, pos, base * net_scale * (nt1 * wn + nt2 * store.net_wns), state['ov'], state['gamma_base'])
    g_wl = wa_grad(d, pos, base, state['ov'], state['gamma_base'])
    return g_pp, g_net, g_wl


def calibrate(d, store, state, pos, t1, t2, cfg, base, target=0.3, lin_scale=1.0, quad_scale=1.0, net_scale=1.0,
              with_net=True, net_t=None):
    """beta from the gradient balance: ||beta (grad PP + grad net)|| / ||grad WL|| = target at this iterate.

    The paper's beta cannot be used as printed (unbounded F_j: report section on units), so its one global beta is set
    from a timing-force budget; the three terms keep the ratios eqs. 17-18 derive. with_net=False leaves the net
    term out of the balance (the earlier --net-bump calibration sets it on its own)."""
    g_pp, g_net, g_wl = timing_grads(d, store, state, pos, t1, t2, cfg, base, lin_scale, quad_scale, net_scale, net_t)
    gt = g_pp + g_net if with_net else g_pp
    nt, nw = float(gt.norm()), float(g_wl.norm())
    info = {'grad_timing': nt, 'grad_wl': nw, 'grad_pp': float(g_pp.norm()), 'grad_net': float(g_net.norm()),
            'target': target}
    if nt <= 0.0:
        return state['beta'], dict(info, ratio_timing_wl=0.0)
    beta = target * nw / max(nt, 1e-30)
    return beta, dict(info, ratio_timing_wl=nt / max(nw, 1e-30))


def target_schedule(path):
    """A beta-target schedule from another run's trace (e.g. DATE25 with its official parameters): the measured
    ||timing force|| / ||grad WL|| (r_time) against the iterations since the timing start, linearly interpolated."""
    tr = json.load(open(path))
    xs = [r['it'] - tr[0]['it'] for r in tr if 'r_time' in r]
    ys = [r['r_time'] for r in tr if 'r_time' in r]

    def f(dt):
        if dt <= xs[0]:
            return ys[0]
        for k in range(1, len(xs)):
            if dt <= xs[k]:
                t = (dt - xs[k - 1]) / max(xs[k] - xs[k - 1], 1)
                return ys[k - 1] + t * (ys[k] - ys[k - 1])
        return ys[-1]
    return f


def calibrate_net(store, t1, t2, bump):
    """beta for the net-level factor from a target multiplier, not a gradient: the 95th percentile of
    beta*(t1 w_tns(e) + t2 w_wns(e)) over the weighted nets equals `bump`. The paper's single beta has the net and
    pin terms in one derived unit; the net factor's own scale (per unit length) is what makes a gradient calibration
    leave the top nets with 50-1000x wirelength weights, so it is calibrated on its own."""
    v = t1 * store.net_tns + t2 * store.net_wns
    m = v > 0
    if not bool(m.any()):
        return 0.0, {'net_bump': bump, 'net_weight_quantile': 0.0}
    q = float(torch.quantile(v[m], 0.95))
    return bump / max(q, 1e-30), {'net_bump': bump, 'net_weight_quantile': q}


def site_width(d):
    """The placement site width (um): DREAMPlace's length unit (PlaceDB scales by 1 / site_width)."""
    return float(d.row_site_w.float().median())


def paper_beta(d, beta, scheme):
    """The paper's beta (DREAMPlace units: lengths in sites, s um each) in Attar's um.

    DREAMPlace minimises WL_site + lambda D + beta PP. With x_site = x_um / s, WL_site = WL_um / s, so multiplying by
    s gives Attar's WL_um + ... + s beta PP. The derived weights are delays (ps) when r, c are per the same length unit
    as the distances, so PP does not depend on the unit: beta_um = s beta. DATE25's weights are unit-free ([10, 50])
    and multiply squared site distances, PP = sum w Q_um / s^2: beta_um = beta / s."""
    s = site_width(d)
    return beta / s if scheme == 'date25' else beta * s


def run(design, method='ours', seed=0, dev=None, root=None, max_iters=2500, start=500, m=20, beta=2.5e-5,
        t1=3e5, t2=1e4, net_cap=None, lg=True, verbose=True, stop_overflow=0.07, beta_target=0.3, diag=True,
        net_scale=1.0, net_bump=None, lin_scale=1.0, quad_scale=1.0, start_ov=None, beta_units='um', gp_kw=None,
        trace_every=None, beta_target_from=None, beta_target_scale=1.0, beta_fixed=False, gp_engine='tdp'):
    """start_ov: start the timing updates at the first iteration whose density overflow is <= start_ov (the paper's
    "iteration 500 of DREAMPlace", where the cells have spread; Fig. 4: overflow ~0.55) instead of at iteration
    `start`. beta_units='site': `beta` is in the paper's (DREAMPlace's) units and is converted (paper_beta).
    trace_every: for the wirelength-only methods, time the placement every k iterations after the start (a trace to
    compare with)."""
    if method not in METHODS:
        raise KeyError('unknown method %r (have %s)' % (method, ', '.join(METHODS)))
    cfg = METHODS[method]
    gp_kw = dict(gp_kw or {})
    if method == 'W1':
        return run_w1(design, seed=seed, dev=dev, root=root, max_iters=max_iters, lg=lg, verbose=verbose,
                      stop_overflow=stop_overflow)
    d = common.load(design, dev, unplace=True)
    base = d.n_weight.clone()
    store = weights.Weights(d, scheme=cfg.get('scheme', 'derived'))
    auto_beta = beta is None or (isinstance(beta, str) and beta == 'auto')
    beta_in = None if auto_beta else float(beta)
    if not auto_beta and beta_units == 'site':
        beta = paper_beta(d, beta_in, cfg.get('scheme', 'derived'))
    go = gpov.GPOverflow(d, bins=gp_kw.get('bins'), target_density=gp_kw.get('target_density', 0.9))
    state = {'ov': 1.0, 'gamma_base': go.gamma_base, 'beta': 2.5e-5 if auto_beta else float(beta),
             'start': None if start_ov is not None else start}
    trace = []
    calib = {}
    sched = target_schedule(beta_target_from) if beta_target_from else None
    t_start = time.time()

    def started(it, pos):
        if state['start'] is None:
            if it % 5:
                return False
            ov = go.overflow(pos)
            if ov > start_ov:
                return False
            state['start'] = it
            if verbose:
                print('  [start] timing from iteration %d (overflow %.3f <= %.3f)' % (it, ov, start_ov), flush=True)
        return it >= state['start']

    def trace_only(it, pos):
        if not started(it, pos) or (it - state['start']) % (trace_every or m):
            return False
        r = d.timing(pos)
        trace.append({'it': int(it), 'tns': float(r.tns), 'wns': float(r.wns), 'hpwl': common.hpwl_unweighted(d, pos),
                      'ov': go.overflow(pos)})
        return False

    def callback(it, pos):
        if not started(it, pos):
            return False
        state['ov'] = go.overflow(pos)         # the timing terms' WA gamma follows the GP's (gpov.py)
        if (it - state['start']) % m:
            return False
        t0 = time.time()
        res = d.timing(pos)
        st = store.update(res)
        if auto_beta and not store.empty() and not (beta_fixed and state.get('fixed')):
            # beta_fixed (the paper's dynamics): calibrate once, at the first update, and keep beta; the cumulative
            # weights then ramp the timing force up update by update, as a constant beta does in the paper.
            # Otherwise recalibrate every update to hold the force at the target (schedule).
            state['fixed'] = True
            tgt = (sched(it - state['start']) if sched else beta_target) * beta_target_scale
            b_new, info = calibrate(d, store, state, pos, tt1, tt2, cfg, base, tgt, lin_scale, quad_scale, net_scale,
                                    with_net=net_bump is None, net_t=(t1, t2))
            state['beta'] = b_new
            share = cfg.get('net_share')
            if share is not None and info['grad_pp'] > 0 and info['grad_net'] > 0:
                # split the budget: ||beta g_pp|| = (1 - share) target ||g_wl||, ||g_net|| = share target ||g_wl||
                state['beta'] = (1.0 - share) * tgt * info['grad_wl'] / info['grad_pp']
                state['net_coef'] = share * tgt * info['grad_wl'] / info['grad_net'] * net_scale
                info.update(beta_split=state['beta'], net_coef=state['net_coef'])
            calib.setdefault('history', []).append({'it': int(it), 'beta': state['beta'], **info})
            calib.update({k: v for k, v in info.items()})
            if verbose:
                print('  [calib] beta %.3e  (grad pp %.3g net %.3g / wl %.3g, target %.3g)'
                      % (state['beta'], info['grad_pp'], info['grad_net'], info['grad_wl'], tgt), flush=True)
        if cfg.get('net_ratio') is not None and not store.empty():
            # D25N: pairs keep state['beta'] (the official one); net coefficient from the measured pair force
            g_pp, g_net, g_wl = timing_grads(d, store, state, pos, tt1, tt2, cfg, base, lin_scale, quad_scale,
                                             net_scale, net_t=(t1, t2))
            npp = float(g_pp.norm()) * state['beta']
            if float(g_net.norm()) > 0:
                state['net_coef'] = cfg['net_ratio'] * npp / float(g_net.norm()) * net_scale
            st['r_pp_d25'] = npp / max(float(g_wl.norm()), 1e-30)
        if cfg.get('net_literal') is not None:
            # paper beta in site units -> um (beta * s), derived weights in ps -> ns (x 1e-3)
            state["net_coef"] = float(beta_in) * site_width(d) * cfg['net_literal'] * net_scale
            if not store.empty():
                g_pp, g_net, g_wl = timing_grads(d, store, state, pos, tt1, tt2, cfg, base, lin_scale, quad_scale,
                                                 net_scale, net_t=(t1, t2))
                st['r_pp_d25'] = float(g_pp.norm()) * state['beta'] / max(float(g_wl.norm()), 1e-30)
                st['r_net'] = float(g_net.norm()) * state['net_coef'] / max(float(g_wl.norm()), 1e-30)
        if cfg['use_net']:
            w = store.net_weight(base, state.get('net_coef', state['beta'] * net_scale), t1, t2, cfg['skip_b'])
            if net_bump is not None:
                bn, ninfo = calibrate_net(store, t1, t2, net_bump)
                state['beta_net'] = bn
                calib['net'] = ninfo
                w = base * (1.0 + bn * (t1 * store.net_tns + t2 * store.net_wns))
            if net_cap:
                w = w.clamp(max=base * net_cap)
            state['net_excess'] = w - base
            if gp_engine == 'tdp':
                d.n_weight.copy_(w)            # gp_tdp reads it after the callback (and preconditions with it)
            else:
                state['net_extra'] = w - base  # Attar's GP: through `extra` (it never reads d.n_weight again)
            mult = (w / base).nan_to_num()
            st['wmult_max'] = float(mult.max().item())
            st['wmult_mean'] = float(mult.mean().item())
            st['wmult_p99'] = float(torch.quantile(mult[mult > 1.0], 0.99)) if bool((mult > 1.0).any()) else 1.0
        if diag and not store.empty():
            # the timing force actually applied at this update, relative to the plain wirelength's gradient
            v_t, g_pp = objective.value_at(d, store, pos, tt1, tt2, cfg['lin'], cfg['quad'], state['ov'],
                                           state['gamma_base'], lin_scale, quad_scale)
            g_pp = g_pp * state['beta']
            g_net = (wa_grad(d, pos, state['net_excess'], state['ov'], state['gamma_base'])
                     if state.get('net_excess') is not None else torch.zeros_like(pos))
            nw = float(wa_grad(d, pos, base, state['ov'], state['gamma_base']).norm())
            st.update(tim_val=state['beta'] * v_t, wl_val=common.hpwl_unweighted(d, pos), beta=state['beta'],
                      r_time=float((g_pp + g_net).norm()) / nw, r_pp=float(g_pp.norm()) / nw,
                      r_net=float(g_net.norm()) / nw)
        rec = {'it': int(it), 'tns': float(res.tns), 'wns': float(res.wns), 'hpwl': common.hpwl_unweighted(d, pos),
               'ov': state['ov'], 'seconds': round(time.time() - t0, 2), **st}
        trace.append(rec)
        if verbose:
            print('  [%s %s s%d] it %4d  TNS %.4g  WNS %.4g  HPWL %.4g  pairs %d  r_time %.3g (pp %.3g net %.3g)  '
                  'wmult p99 %.3g max %.3g  (%.1fs)'
                  % (method, design, seed, it, rec['tns'], rec['wns'], rec['hpwl'], st.get('pairs', 0),
                     st.get('r_time', 0.0), st.get('r_pp', 0.0), st.get('r_net', 0.0), st.get('wmult_p99', 1.0),
                     st.get('wmult_max', 1.0), rec['seconds']), flush=True)
        return False

    uses_timing = cfg['lin'] or cfg['quad'] or cfg['use_net']
    tt1, tt2 = (1.0, 0.0) if cfg.get('scheme') == 'date25' else (t1, t2)   # DATE25 has no t1/t2 split
    extra = (objective.make_extra(d, store, state, tt1, tt2, cfg['lin'], cfg['quad'], lin_scale, quad_scale)
             if uses_timing else None)
    cb = callback if uses_timing else (trace_only if trace_every else None)
    place = gp_tdp.global_place if gp_engine == 'tdp' else naive.global_place
    gp = place(d, seed=seed, max_iters=max_iters, stop_overflow=stop_overflow, extra=extra,
                            callback=cb, callback_every=1, verbose=verbose, **gp_kw)
    d.n_weight.copy_(base)
    hist = gp.pop('history', None) or {}
    gp['seconds'] = round(gp.get('seconds', time.time() - t_start), 1)
    gp['start_iteration'] = state['start']
    gp['overflow_every10'] = [round(v, 4) for v in hist.get('overflow', [])[::10]]
    pre_lg = common.measure(d, legality=False)
    gp_pos = d.pos.detach().clone()
    if lg:
        t0 = time.time()
        for _ in range(3):                      # a first pass can leave a few tall cells outside the rows: the next
            naive.legalize(d)                   # one runs incrementally and only moves them
            if Evaluator(d).legality()['legal']:
                break
        gp['legalize_seconds'] = round(time.time() - t0, 1)
    metrics = common.measure(d)
    metrics['method'] = method
    metrics['design'] = design
    metrics['seed'] = seed
    metrics['gp'] = gp
    metrics['pre_lg'] = pre_lg
    metrics['config'] = {'max_iters': max_iters, 'start': start, 'start_ov': start_ov, 'm': m,
                         'beta': state['beta'] if auto_beta else beta, 'beta_arg': beta_in, 'beta_units': beta_units,
                         'site_um': site_width(d), 'gp_kw': gp_kw, 'stop_overflow': stop_overflow,
                         't1': t1, 't2': t2, 'beta_target': beta_target, 'beta_target_from': beta_target_from, 'beta_target_scale': beta_target_scale, 'beta_fixed': beta_fixed, 'gp_engine': gp_engine,
                         'net_cap': net_cap, 'net_scale': net_scale, 'net_bump': net_bump,
                         'lin_scale': lin_scale, 'quad_scale': quad_scale,
                         'device': str(d.device),
                         'attar': attar.__version__}
    metrics['calib'] = calib
    metrics['runtime_s'] = round(time.time() - t_start, 1)
    metrics['weights'] = {k: v for k, v in store.stats.items()}
    if root is not False:
        common.save_run(common.run_dir(method, design, seed, root), metrics['config'], metrics, trace,
                        d.pos if lg else None, gp_pos=gp_pos)
    return metrics, trace, store


def run_w1(design, seed=0, dev=None, root=None, max_iters=2500, lg=True, verbose=True, stop_overflow=0.07):
    """OpenROAD gpl's timing-driven global placement as Attar's example does it: place, weight the worst 10 % of the
    nets by their net slack (weights 1..5), place again from scratch."""
    d = common.load(design, dev, unplace=True)
    base = d.n_weight.clone()
    t_start = time.time()
    naive.global_place(d, seed=seed, max_iters=max_iters, stop_overflow=stop_overflow, verbose=verbose)
    if lg:
        naive.legalize(d)
    s = d.timing().net_slack()
    v = s[s.isfinite()].sort().values
    v = v[:max(1, len(v) // 10)]
    if len(v) < 2:
        w = base.clone()
    else:
        lo, hi = float(v[len(v) // 50]), float(v[-1])
        crit = ((hi - s) / (hi - lo)).clamp(0, 1) if hi > lo else torch.zeros_like(s)
        w = base * (1 + 4 * crit.clamp(0, 1))
    d.n_weight.copy_(w)
    d.unplace()
    gp2 = naive.global_place(d, seed=seed, max_iters=max_iters, stop_overflow=stop_overflow, verbose=verbose)
    d.n_weight.copy_(base)
    if lg:
        naive.legalize(d)
    metrics = common.measure(d)
    metrics.update(method='W1', design=design, seed=seed, gp={'pass2': {k: v for k, v in gp2.items() if k != 'history'}},
                   config={'max_iters': max_iters, 'device': str(d.device), 'attar': attar.__version__},
                   runtime_s=round(time.time() - t_start, 1))
    if root is not False:
        common.save_run(common.run_dir('W1', design, seed, root), metrics['config'], metrics, None, d.pos)
    return metrics, None, None

In [ ]:
%%writefile src/gp_tdp.py
"""P11's copy of Attar 1.0.4's naive.global_place (attar/naive/gp.py, main 6e3bb3f) with the two behaviours of
DREAMPlace, the paper's (and DATE25-TDP's) placer, that net-weighted timing-driven placement relies on:

  live_net_weights  the wirelength reads d.n_weight as it is after every callback (Attar copies it into the
                    Evaluator's wmask once, at the start, so in-place changes never reached the placer)
  precond='net_weight'  the per-cell preconditioner is the sum of the net weights of the cell's pins (+ lambda x
                    area), as DREAMPlace's PreconditionOp (sum_pin_weights_in_nodes = pws_op(net_weights)); Attar's
                    is the pin count. A heavily weighted net's cells then take proportionally smaller steps.
With the netlist's own weights both are no-ops: the trajectory is bit-identical to Attar's (tests/test_gp_tdp.py).
Everything else is Attar's code unchanged.

Attar's original docstring:
Naive global placement: ePlace-style electrostatic placement (after ePlace, DREAMPlace and Xplace), pure PyTorch.

  objective   WA wirelength + lambda * electrostatic energy of the cell density
  density     cell area spread exactly over bins (attar.kernels.Grid.accumulate); cells smaller than sqrt(2) bins
              are expanded to that size with proportionally lower density, as in ePlace
  field       Poisson equation solved with the DCT, written as matrix products with cos/sin bases (cuBLAS)
  gradient    WA by autograd; density gradient = -(charge density) x integral of the field over the cell
              (attar.kernels.Grid.integrate), exact for the piecewise-constant field
  optimizer   Nesterov with Lipschitz step estimation and backtracking; per-cell preconditioner
  schedule    lambda grows by mu each iteration (DREAMPlace rule), WA gamma follows the overflow, stop at overflow
              below `stop_overflow`
Fillers occupy the white space up to `target_density`. Fixed cells and macros are static charge. Movable macros are
treated as big cells (no macro legalization here).
"""
import math
import time

import torch

from attar.io.distill import ASIDE, MACRO, TERMINAL
from attar import metrics as M
from attar.kernels import Grid, default_bins


def _graph(fn, example):
    """Capture fn(x) as a CUDA graph; returns f(x) that copies x into the static input and replays."""
    static = example.detach().clone()
    dev = example.device                           # capture on the tensors' GPU, not the current default one
    with torch.cuda.device(dev):
        s = torch.cuda.Stream(dev); s.wait_stream(torch.cuda.current_stream(dev))
        with torch.cuda.stream(s):
            for _ in range(2):
                fn(static)
        torch.cuda.current_stream(dev).wait_stream(s)
        graph = torch.cuda.CUDAGraph()
        with torch.cuda.graph(graph, stream=s):     # torch's default capture stream is created once, on one GPU
            out = fn(static)

    def run(x):
        static.copy_(x)
        with torch.cuda.device(dev):
            graph.replay()
        return out
    return run



# DREAMPlace's routability defaults (dreamplace/params.json), RUDY branch
ROUTE_DEFAULTS = {'node_area_adjust_overflow': 0.15, 'max_num_area_adjust': 3, 'route_opt_adjust_exponent': 2.0,
                  'max_route_opt_adjust_rate': 2.0, 'area_adjust_stop_ratio': 0.01, 'route_area_adjust_stop_ratio': 0.01,
                  'routing_layers': None}


def _track_capacity(d, g, ra):
    """Routing tracks per um (horizontal, vertical) of every bin: 1/pitch summed over the signal routing layers of
    each direction (ra['routing_layers'] = (lowest, highest) layer names; default: the source flow's own, recorded in
    manifest['routing'], else all but the lowest), times the share the router may use (manifest['routing']
    ['adjustment'] held back), less the layers up to the macros' highest pin layer over the part of a bin macros
    cover."""
    import os
    from ..io import st
    tp, _ = st.load(os.path.join(d.path, 'tech_profile.safetensors'))
    names = d.names['layers']
    rt = d.manifest.get('routing', {})                     # the source flow's own routing resources, when recorded
    rng = ra.get('routing_layers') or rt.get('layers')
    lo_l, hi_l = (names.index(rng[0]), names.index(rng[1])) if rng else (1, len(names) - 1)
    keep = 1.0 - float(rt.get('adjustment', 0.0))          # share of a layer's tracks the router may use
    pitch = tp['layer_pitch'].astype('float64') / d.dbu
    dirs = tp['layer_dir']
    layers = range(lo_l, hi_l + 1)
    dev = d.device
    # macros: up to their highest signal-pin layer they block routing over their area
    mac = d.macros
    top = -1
    if len(mac):
        mp = d.raw['m_pin_layer']; ptr = d.raw['m_pin_ptr']
        for m in set(d.raw['c_master'][mac.cpu().numpy()].tolist()):
            if m >= 0 and ptr[m + 1] > ptr[m]:
                top = max(top, int(mp[ptr[m]:ptr[m + 1]].max()))
    cov = torch.zeros(g.nx, g.ny, device=dev)
    if len(mac) and top >= 0:
        p = d.pos[mac]; s = d.size[mac]
        cov = (g.accumulate(p[:, 0], p[:, 1], p[:, 0] + s[:, 0], p[:, 1] + s[:, 1], torch.ones(len(mac), device=dev))
               / g.bin_area).clamp(max=1.0)
    cap = []
    for horizontal in (True, False):
        free = keep * sum(1.0 / pitch[l] for l in layers if (dirs[l] == 0) == horizontal and pitch[l] > 0 and l > top)
        blocked = keep * sum(1.0 / pitch[l] for l in layers if (dirs[l] == 0) == horizontal and pitch[l] > 0 and l <= top)
        cap.append(free + blocked * (1 - cov))
    return cap[0] if torch.is_tensor(cap[0]) else torch.full((g.nx, g.ny), cap[0], device=dev), \
        cap[1] if torch.is_tensor(cap[1]) else torch.full((g.nx, g.ny), cap[1], device=dev)

class Electrostatics:
    """Density map -> potential and field on an nx x ny grid, via DCT bases as matrices."""

    def __init__(self, grid):
        g = grid
        dev = g.device
        self.g = g
        ix = torch.arange(g.nx, device=dev, dtype=torch.float64) + 0.5
        iy = torch.arange(g.ny, device=dev, dtype=torch.float64) + 0.5
        u = torch.arange(g.nx, device=dev, dtype=torch.float64)
        v = torch.arange(g.ny, device=dev, dtype=torch.float64)
        self.Cx = torch.cos(math.pi * u[:, None] * ix[None, :] / g.nx).float()      # (u, i)
        self.Cy = torch.cos(math.pi * v[:, None] * iy[None, :] / g.ny).float()
        self.Sx = torch.sin(math.pi * u[:, None] * ix[None, :] / g.nx).float()
        self.Sy = torch.sin(math.pi * v[:, None] * iy[None, :] / g.ny).float()
        wx = (math.pi * u / (g.nx * g.bw)).float()                                  # physical frequencies
        wy = (math.pi * v / (g.ny * g.bh)).float()
        w2 = wx[:, None] ** 2 + wy[None, :] ** 2
        w2[0, 0] = 1.0
        inv = 1.0 / w2
        inv[0, 0] = 0.0                                                             # no DC term
        norm = torch.full((g.nx, g.ny), 4.0 / (g.nx * g.ny), device=dev)
        norm[0, :] *= 0.5; norm[:, 0] *= 0.5
        self.P = norm * inv                                  # potential coefficients
        self.Px = norm * inv * wx[:, None]                   # field x
        self.Py = norm * inv * wy[None, :]                   # field y

    def __call__(self, rho):
        a = self.Cx @ rho @ self.Cy.T                        # DCT-II coefficients (unnormalised)
        psi = self.Cx.T @ (a * self.P) @ self.Cy
        ex = self.Sx.T @ (a * self.Px) @ self.Cy
        ey = self.Cx.T @ (a * self.Py) @ self.Sy
        return psi, ex, ey


def _diverged(hist, best, window=50, threshold=0.05):
    """DREAMPlace's divergence check (NonLinearPlace.check_divergence): once the overflow has been below 0.25, the
    last `window` iterations average an HPWL more than 1.2 x threshold above the best solution's and an overflow more
    than threshold above the best overflow."""
    if best[0] > 0.25 or len(hist['overflow']) < window:
        return False
    wl = sum(hist['hpwl'][-window:]) / window
    ov = sum(hist['overflow'][-window:]) / window
    return (wl - best[1]) / best[1] > threshold * 1.2 and (ov - best[0]) / max(best[0], 1e-9) > threshold


def global_place(d, target_density=0.9, stop_overflow=0.07, max_iters=2000, bins=None, seed=0,
                 verbose=True, log_every=50, snapshot='naive_gp', graphs=True, movable=None, macro_halo=0.0,
                 routability=False, route_args=None, extra=None, callback=None, callback_every=1,
                 live_net_weights=True, precond='net_weight'):
    """Place the movable cells of d. Sets d.pos (the placement becomes a fact) and saves a snapshot.
    Returns a dict with the trajectory (iterations, HPWL, overflow, time).

    Hooks for a user's objective terms (timing-driven placement and the like):
      extra(pos) -> (value, grad): a term added to the objective at every evaluation; pos (C, 2) the lower-left of
        every cell (as d.pos; the movable ones at the iterate), grad (C, 2) its gradient with respect to pos (only
        the movable cells' rows are used). Added after the wirelength and density, with the same preconditioner (the
        cell's pin count + lambda x area). Called outside the CUDA graph: it may sync, call d.timing(pos), etc.
      callback(it, pos) every callback_every iterations, after the step: e.g. time the placement (d.timing(pos)) and
        update d.n_weight in place (the wirelength reads it at the next evaluation) or the extra term's state.
        Returning True stops the placement there.
    Net weights (d.n_weight) weigh the wirelength as they are when each evaluation runs."""
    from attar.evaluator import Evaluator
    if movable is None:
        d.require('global_place')
    dev = d.device
    t_start = time.time()
    torch.manual_seed(seed)
    if movable is None:
        util = float(d.utilization()[0])
        if util > target_density:           # cells cannot spread thinner than they are (ISPD 2019 test7-10: 96%)
            if verbose:
                print('  gp: utilization %.3f > target density %.3f: target raised to it' % (util, target_density))
            target_density = util
    core = d.core_box[0]
    cx0, cy0, cx1, cy1 = [float(v) for v in core]
    W, H = cx1 - cx0, cy1 - cy0

    mvmask = d.movable if movable is None else movable
    mv = torch.nonzero(mvmask).flatten()
    fx = torch.nonzero(~mvmask & ((d.flags & (TERMINAL | ASIDE)) == 0)).flatten()
    size = d.size
    msize = size[mv]
    marea = msize.prod(1)
    mov_area = marea.sum()

    # fixed objects: static density (clipped to the bin), free area
    n_bins = bins or default_bins(len(mv))
    g = Grid(cx0, cy0, cx1, cy1, n_bins, n_bins, dev)
    fix_map = torch.zeros(g.nx, g.ny, device=dev)
    if len(fx):
        p = d.pos[fx]
        # macro_halo: a keep-out band (um) around every fixed macro, counted as taken
        h = (macro_halo * ((d.flags[fx] & MACRO) != 0).float())[:, None] if macro_halo else torch.zeros(len(fx), 1, device=dev)
        x0 = (p[:, 0:1] - h).clamp(min=cx0)[:, 0]; y0 = (p[:, 1:2] - h).clamp(min=cy0)[:, 0]
        x1 = (p[:, 0:1] + size[fx, 0:1] + h).clamp(max=cx1)[:, 0]; y1 = (p[:, 1:2] + size[fx, 1:2] + h).clamp(max=cy1)[:, 0]
        fix_map = g.accumulate(x0, y0, x1, y1, torch.ones(len(fx), device=dev)).clamp(max=g.bin_area)
    # area without rows (e.g. macro halos and channels where the flow cut the rows) cannot take cells either
    if d.row_box is not None:
        hb = d.row_h.min()
        base = d.row_h <= hb + 1e-6
        rb = d.row_box[base]
        rows_cov = g.accumulate(rb[:, 0], rb[:, 1], rb[:, 2], rb[:, 3], torch.ones(len(rb), device=dev)).clamp(max=g.bin_area)
        fix_map = torch.maximum(fix_map, g.bin_area - rows_cov).clamp(max=g.bin_area)
    free_area = (g.bin_area - fix_map).sum()

    # fillers: average standard-cell size, fill up to the target density
    std = (d.flags[mv] & MACRO) == 0
    fsize = msize[std].mean(0) if std.any() else msize.mean(0)
    filler_area = (target_density * free_area - mov_area).clamp(min=0)
    n_fill = int((filler_area / fsize.prod()).item())
    nodes_size = torch.cat([msize, fsize[None].expand(n_fill, 2)], 0)
    n_mov = len(mv)

    # expanded sizes for the density (ePlace local smoothing)
    min_sz = torch.tensor([g.bw, g.bh], device=dev) * math.sqrt(2)
    esize = torch.maximum(nodes_size, min_sz)
    ratio = nodes_size.prod(1) / esize.prod(1)               # density per unit area of the expanded cell
    td = torch.full((), float(target_density), device=dev)  # target density: a tensor, raised by cell inflation
    static = fix_map * target_density                         # fixed charge, scaled like Xplace

    es = Electrostatics(g)
    ev = Evaluator(d)
    cell, net, E, wmask = ev.cell, ev.net, ev.E, ev.wmask
    off = ev.off = ev.off.clone()                         # pin offsets from the lower-left: follow cell inflation
    pins = torch.zeros(len(d.flags), device=dev).index_add_(0, d.p_cell, torch.ones_like(d.p_cell, dtype=torch.float32))
    npins = torch.cat([pins[mv], torch.zeros(n_fill, device=dev)])
    npins0 = npins.clone()
    wmask0 = wmask.clone()
    cell_of_mv = torch.full((len(d.flags),), -1, dtype=torch.long, device=dev)
    cell_of_mv[mv] = torch.arange(len(mv), device=dev)

    def refresh_weights():
        """After a callback: the wirelength's net weights from d.n_weight (in place: the CUDA graph reads the same
        tensors), and DREAMPlace's preconditioner, pin count + sum over the cell's pins of (weight - initial weight)."""
        if live_net_weights:
            wmask.copy_(torch.where(ev.deg >= 2, d.n_weight, torch.zeros_like(d.n_weight)))
        if precond == 'net_weight':
            dw = (wmask - wmask0)[net]                        # per pin (grouped by net, as `cell`)
            k = cell_of_mv[cell]
            ok = k >= 0
            add = torch.zeros(n_mov + n_fill, device=dev).index_add_(0, k[ok], dw[ok])
            npins.copy_(npins0 + add)
    area_all = nodes_size.prod(1)

    # initial positions (lower-left): movable cells around the core center, fillers spread uniformly
    ctr = torch.tensor([(cx0 + cx1) / 2, (cy0 + cy1) / 2], device=dev)
    x = torch.empty(n_mov + n_fill, 2, device=dev)
    x[:n_mov] = ctr + torch.randn(n_mov, 2, device=dev) * torch.tensor([W, H], device=dev) * 0.001 - msize / 2
    x[n_mov:] = torch.rand(n_fill, 2, device=dev) * torch.tensor([W, H], device=dev) + torch.tensor([cx0, cy0], device=dev) - fsize / 2
    lo = torch.tensor([cx0, cy0], device=dev)
    hi = torch.tensor([cx1, cy1], device=dev) - nodes_size

    base = d.pos.detach().clone()
    gamma_base = 2.0 * (g.bw + g.bh)
    lam = torch.zeros((), device=dev)                    # tensors, so CUDA graphs see the updates
    gamma = torch.full((), gamma_base * 10.0, device=dev)
    static_rho = static / g.bin_area
    half_e = esize / 2
    mov_area_t = mov_area.clone()

    def full_pos(xm):
        return base.index_copy(0, mv, xm)

    def obj_grad(xa):
        """objective, preconditioned gradient (forward only: analytic WA and density gradients)"""
        xa = torch.maximum(torch.minimum(xa, hi), lo)
        p = full_pos(xa[:n_mov])[cell] + off
        wl, gp = M.wa_value_grad(p, net, E, gamma, wmask)
        gwl = torch.zeros(len(base), 2, device=dev).index_add_(0, cell, gp)[mv]
        cen = xa + nodes_size / 2
        c = g.corners(cen[:, 0] - half_e[:, 0], cen[:, 1] - half_e[:, 1], cen[:, 0] + half_e[:, 0], cen[:, 1] + half_e[:, 1])
        rho = g.accumulate_c(c, ratio) / g.bin_area + static_rho
        psi, ex, ey = es(rho)
        energy = 0.5 * (rho * psi).sum() * g.bin_area
        gd = -g.integrate_c(torch.stack([ex, ey]), c) * ratio[:, None]
        grad = lam * gd
        grad[:n_mov] += gwl
        pre = (npins + lam * area_all).clamp(min=1.0)
        return wl + lam * energy, grad / pre[:, None], gwl, gd

    def metrics(ua):
        hp = ev.hpwl(full_pos(ua[:n_mov]))
        cen = ua[:n_mov] + nodes_size[:n_mov] / 2
        c = g.corners(cen[:, 0] - half_e[:n_mov, 0], cen[:, 1] - half_e[:n_mov, 1], cen[:, 0] + half_e[:n_mov, 0], cen[:, 1] + half_e[:n_mov, 1])
        m = g.accumulate_c(c, ratio[:n_mov])
        ov = (m + fix_map * td - td * g.bin_area).clamp(min=0).sum() / mov_area_t
        return torch.stack([hp, ov])

    # initial lambda: ePlace / DREAMPlace ratio of gradient norms
    _, _, gwl0, gd0 = obj_grad(x)
    lam.fill_((gwl0.abs().sum() / gd0.abs().sum().clamp(min=1e-30) * 8e-5).item())

    # CUDA graphs: one replay per objective evaluation instead of ~100 kernel launches
    og, mt = obj_grad, metrics
    if dev.type == 'cuda' and graphs:
        og = _graph(lambda xa: obj_grad(xa)[:2], x)
        mt = _graph(metrics, x)
    else:
        og = lambda xa: obj_grad(xa)[:2]
    if extra is not None:                               # the user's term, outside the graph, same preconditioner
        og_base = og

        def og(xa):
            f, gr = og_base(xa)
            xa = torch.maximum(torch.minimum(xa, hi), lo)
            ve, ge = extra(full_pos(xa[:n_mov]).detach())
            gr = gr.clone()
            pre = (npins[:n_mov] + lam * area_all[:n_mov]).clamp(min=1.0)
            gr[:n_mov] += ge[mv].to(gr.dtype) / pre[:, None]
            return f + ve, gr

    # WA gamma from the overflow, from the start (DREAMPlace): a fixed initial value jumps at the first update when
    # the initial overflow is far from 1 (ispd2006_hb/ibm02: 0.58), and the Lipschitz estimate then collapses
    gamma_of = lambda ov: gamma_base * 10 ** ((min(max(ov, 0.0), 1.0) - 0.1) * 20 / 9 - 1)
    gamma.fill_(gamma_of(float(metrics(x)[1])))

    # routability (DREAMPlace adjust_node_area, RUDY branch): when the overflow first falls below
    # node_area_adjust_overflow, cells in bins whose routing utilization (RUDY over track capacity) is high grow by
    # clamp(util^exponent, 1/max_rate, max_rate) of their area, within 10% of the whitespace; the target density
    # follows; at most max_num_area_adjust times, until the growth is below area_adjust_stop_ratio
    ra = dict(ROUTE_DEFAULTS, **(route_args or {}), active=routability, done=0, rounds=[])
    if routability:
        cap_h, cap_v = _track_capacity(d, g, ra)
        ra['capacity_h'], ra['capacity_v'] = float(cap_h.mean()), float(cap_v.mean())
        total_place = free_area                                  # area cells may use
        std_mov = torch.zeros(n_mov + n_fill, dtype=torch.bool, device=dev); std_mov[:n_mov] = std
        dl = torch.zeros(len(base), 2, device=dev)

    def inflate(ua):
        """Grow the cells in congested bins (in place: sizes, density terms, pin offsets, positions). True if any grew."""
        p = full_pos(ua[:n_mov])[cell] + off
        lo_, hi_ = M.net_bbox(p, net, E)
        ok = ev.deg >= 2
        lo_, hi_, w_ = lo_[ok], hi_[ok], wmask[ok]
        sz = torch.maximum(hi_ - lo_, torch.tensor([g.bw, g.bh], device=dev))
        c_ = (lo_ + hi_) / 2; lo_, hi_ = c_ - sz / 2, c_ + sz / 2
        dem_h = g.accumulate(lo_[:, 0], lo_[:, 1], hi_[:, 0], hi_[:, 1], w_ / sz[:, 1]) / g.bin_area   # horizontal tracks/um
        dem_v = g.accumulate(lo_[:, 0], lo_[:, 1], hi_[:, 0], hi_[:, 1], w_ / sz[:, 0]) / g.bin_area
        util = torch.maximum(dem_h / cap_h.clamp(min=1e-9), dem_v / cap_v.clamp(min=1e-9))
        adj = util.pow(ra['route_opt_adjust_exponent']).clamp(1 / ra['max_route_opt_adjust_rate'], ra['max_route_opt_adjust_rate'])
        cen = ua + nodes_size / 2
        cc = g.corners(cen[:, 0] - nodes_size[:, 0] / 2, cen[:, 1] - nodes_size[:, 1] / 2,
                       cen[:, 0] + nodes_size[:, 0] / 2, cen[:, 1] + nodes_size[:, 1] / 2)
        route_area = g.integrate_c(torch.stack([adj, adj]), cc)[:, 0]          # sum over bins of overlap x adj
        old = nodes_size.prod(1)
        inc = torch.where(std_mov, (route_area - old).clamp(min=0), torch.zeros_like(old))
        old_mov = old[:n_mov].sum(); inc_sum = inc.sum()
        ratio_inc = float(inc_sum / old_mov)
        ra['rounds'].append({'iteration': len(hist['iter']), 'max_util': float(util.max()), 'growth': ratio_inc})
        if ratio_inc < ra['area_adjust_stop_ratio'] or ratio_inc < ra['route_area_adjust_stop_ratio']:
            ra['active'] = False
            return False
        budget = min(0.1 * float(total_place - old_mov), float(total_place - old_mov))
        scale = budget / float(inc_sum)
        if scale <= 0:
            ra['active'] = False
            return False
        inc = inc * min(1.0, scale)
        k = ((old + inc) / old).sqrt()
        new_size = nodes_size * k[:, None]
        new_mov = new_size[:n_mov].prod(1).sum()
        fill_old = old[n_mov:].sum()
        if n_fill and new_mov + fill_old > total_place * td:     # fillers give the room back
            fr = ((total_place * td - new_mov).clamp(min=0) / fill_old).sqrt()
            new_size[n_mov:] = new_size[n_mov:] * fr
        delta = (new_size - nodes_size) / 2                      # keep centers: lower-left moves by -delta
        ua.sub_(delta)
        dl.zero_(); dl[mv] = delta[:n_mov]
        off.add_(dl[cell])                                       # pins keep their absolute place
        nodes_size.copy_(new_size)
        esize.copy_(torch.maximum(nodes_size, min_sz)); half_e.copy_(esize / 2)
        ratio.copy_(nodes_size.prod(1) / esize.prod(1)); area_all.copy_(nodes_size.prod(1))
        hi.copy_(torch.tensor([cx1, cy1], device=dev) - nodes_size)
        mov_area_t.copy_(nodes_size[:n_mov].prod(1).sum())
        td.copy_(((nodes_size.prod(1).sum()) / total_place).clamp(max=1.0))
        static_rho.copy_(fix_map * td / g.bin_area)
        ra['done'] += 1
        ra['rounds'][-1].update(scale=min(1.0, scale), target_density=float(td))
        return True

    # Nesterov (ePlace Algorithm 2) with Lipschitz step estimation and backtracking
    u = x.clone()
    v = x.clone()
    f_v, g_v = [t.clone() for t in og(v)]
    v1 = torch.maximum(torch.minimum(v - 0.01 * min(g.bw, g.bh) * g_v / g_v.norm().clamp(min=1e-30) * math.sqrt(len(v)), hi), lo)
    _, g_v1 = [t.clone() for t in og(v1)]
    alpha = ((v - v1).norm() / (g_v - g_v1).norm().clamp(min=1e-30)).item()
    a_k = 1.0
    hist = {'iter': [], 'hpwl': [], 'overflow': [], 'lambda': [], 'gamma': []}
    prev_hpwl = None
    best = None                                              # (overflow, HPWL, iteration, positions)
    diverged = False
    it = 0
    for it in range(max_iters):
        a_k1 = (1 + math.sqrt(4 * a_k * a_k + 1)) / 2
        beta = (a_k - 1) / a_k1
        alpha0 = alpha
        for _tries in range(10):                               # backtracking on the Lipschitz estimate
            u1 = torch.maximum(torch.minimum(v - alpha * g_v, hi), lo)
            v_new = torch.maximum(torch.minimum(u1 + beta * (u1 - u), hi), lo)
            f_new, g_new = [t.clone() for t in og(v_new)]
            alpha_new = ((v_new - v).norm() / (g_new - g_v).norm().clamp(min=1e-30)).item()
            if alpha_new > 0.95 * alpha:
                break
            alpha = alpha_new
        else:
            # no step accepted: g_v predates this iteration's lambda / gamma update; take it again here and step once
            g_v = og(v)[1].clone()
            alpha = alpha0
            u1 = torch.maximum(torch.minimum(v - alpha * g_v, hi), lo)
            v_new = torch.maximum(torch.minimum(u1 + beta * (u1 - u), hi), lo)
            f_new, g_new = [t.clone() for t in og(v_new)]
            alpha_new = ((v_new - v).norm() / (g_new - g_v).norm().clamp(min=1e-30)).item()
        u, v, g_v, alpha, a_k = u1, v_new, g_new, alpha_new, a_k1

        # schedule (one host sync for HPWL and overflow)
        hp, ov = mt(u).tolist()
        if prev_hpwl is not None:
            dh = hp - prev_hpwl
            mu = 1.05 * max(0.9999 ** it, 0.98) if dh < 0 else 1.05 * min(max(1.05 ** (-dh / 350000), 0.95), 1.05)
            lam.mul_(mu)
        prev_hpwl = hp
        gamma.fill_(gamma_of(ov))
        hist['iter'].append(it); hist['hpwl'].append(hp); hist['overflow'].append(ov)
        if verbose and (it % log_every == 0):
            print('  gp iter %4d  HPWL %.4e  overflow %.3f  lambda %.2e  gamma %.3g' % (it, hp, ov, lam.item(), gamma.item()))
        if routability and ra['active'] and ov < ra['node_area_adjust_overflow'] and ra['done'] < ra['max_num_area_adjust']:
            if inflate(u):                                   # cells grew: restart the momentum from u
                u, v = u.clone(), u.clone()
                f_v, g_v = [t.clone() for t in og(v)]
                a_k = 1.0
                best = None
                prev_hpwl = None
                continue
        if callback is not None and it % max(1, int(callback_every)) == 0:
            stop = callback(it, full_pos(u[:n_mov]).detach())
            refresh_weights()
            if stop:
                hist['stopped_by_callback'] = it
                if best is None or ov < best[0]:
                    best = (ov, hp, it, u.clone())
                break
        if best is None or ov < best[0]:
            best = (ov, hp, it, u.clone())
        if ov < stop_overflow and it > 20:
            break
        if _diverged(hist, best):                          # roll back to the best solution (DREAMPlace)
            diverged = True
            break

    if dev.type == 'cuda':
        torch.cuda.synchronize()
    if best is not None and best[0] < hist['overflow'][-1] - 1e-9:
        u = best[3]
        hist['restored'] = best[2]
    if routability:                                          # back to the true sizes, centers kept
        u = u.clone(); u[:n_mov] += (nodes_size[:n_mov] - msize) / 2
    if movable is None:
        d.pos = full_pos(u[:n_mov]).detach()
        d.snapshot(snapshot)
    else:                                          # mixed-size use (macro placer): positions only, no state change
        d._pos = full_pos(u[:n_mov]).detach()
    k = hist.get('restored', -1)
    res = {'iterations': it + 1, 'hpwl': hist['hpwl'][k], 'overflow': hist['overflow'][k], 'diverged': diverged,
           'restored_iteration': hist.get('restored'),
           'seconds': time.time() - t_start, 'bins': (g.nx, g.ny), 'fillers': n_fill, 'history': hist}
    if routability:
        res['routability'] = {k: v for k, v in ra.items() if k in ('done', 'rounds', 'capacity_h', 'capacity_v')}
    if verbose:
        print('  gp done: %d iterations, HPWL %.4e, overflow %.3f, %.2fs (%.2f ms/iter), %d bins, %d fillers'
              % (res['iterations'], res['hpwl'], res['overflow'], res['seconds'], 1e3 * res['seconds'] / res['iterations'],
                 g.nx, n_fill))
    return res

In [ ]:
%%writefile src/gpov.py
"""The density overflow and WA gamma exactly as naive.global_place computes them (attar/naive/gp.py), so the timing
terms' WA smoothing follows the placer's gamma and the timing start follows the placer's own overflow (the paper's
Fig. 4 overflow is the placer's).

Evaluator.density_overflow differs late in the run (superblue18, 512 bins: 0.18-0.26 against the GP's 0.09) because
the GP spreads each cell over at least sqrt(2) bins (ePlace local smoothing) and counts row-less area as taken.
"""
import math

import torch

from attar.io.distill import ASIDE, TERMINAL
from attar.kernels import Grid, default_bins


class GPOverflow:
    def __init__(self, d, bins=None, target_density=0.9):
        dev = d.device
        mv = torch.nonzero(d.movable).flatten()
        fx = torch.nonzero(~d.movable & ((d.flags & (TERMINAL | ASIDE)) == 0)).flatten()
        util = float(d.utilization()[0])
        td = max(float(target_density), util)
        cx0, cy0, cx1, cy1 = [float(v) for v in d.core_box[0]]
        g = Grid(cx0, cy0, cx1, cy1, bins or default_bins(len(mv)), bins or default_bins(len(mv)), dev)
        fix_map = torch.zeros(g.nx, g.ny, device=dev)
        if len(fx):
            p = d.pos[fx]
            size = d.size[fx]
            fix_map = g.accumulate(p[:, 0].clamp(min=cx0), p[:, 1].clamp(min=cy0), (p[:, 0] + size[:, 0]).clamp(max=cx1),
                                   (p[:, 1] + size[:, 1]).clamp(max=cy1),
                                   torch.ones(len(fx), device=dev)).clamp(max=g.bin_area)
        if d.row_box is not None:
            hb = d.row_h.min()
            rb = d.row_box[d.row_h <= hb + 1e-6]
            rows_cov = g.accumulate(rb[:, 0], rb[:, 1], rb[:, 2], rb[:, 3],
                                    torch.ones(len(rb), device=dev)).clamp(max=g.bin_area)
            fix_map = torch.maximum(fix_map, g.bin_area - rows_cov).clamp(max=g.bin_area)
        msize = d.size[mv]
        min_sz = torch.tensor([g.bw, g.bh], device=dev) * math.sqrt(2)
        esize = torch.maximum(msize, min_sz)
        self.g, self.mv, self.td, self.fix_map = g, mv, td, fix_map
        self.half_e, self.msize = esize / 2, msize
        self.ratio = msize.prod(1) / esize.prod(1)
        self.mov_area = msize.prod(1).sum()
        self.gamma_base = 2.0 * (g.bw + g.bh)

    def overflow(self, pos):
        g = self.g
        u = pos[self.mv]
        cen = u + self.msize / 2
        c = g.corners(cen[:, 0] - self.half_e[:, 0], cen[:, 1] - self.half_e[:, 1],
                      cen[:, 0] + self.half_e[:, 0], cen[:, 1] + self.half_e[:, 1])
        m = g.accumulate_c(c, self.ratio)
        return float((m + self.fix_map * self.td - self.td * g.bin_area).clamp(min=0).sum() / self.mov_area)

    def gamma(self, ov):
        ov = min(max(float(ov), 0.0), 1.0)
        return self.gamma_base * 10 ** ((ov - 0.1) * 20 / 9 - 1)

In [ ]:
%%writefile src/objective.py
"""PP1, PP2 (paper eqs. 20-21) and the net-level factor (eq. 19's first term) as an `extra` term of naive.global_place.

PP1 = sum_(i,j) (t1 w_tns(L_ij) + t2 w_wns(L_ij)) * L^WA_ij,  PP2 = sum_(i,j) (t1 w_tns(L^2_ij) + t2 w_wns(L^2_ij)) * Q_ij

L^WA_ij is the weighted-average distance of the two pins alone (the same model as the placer's net wirelength: for a
two-pin "net" it is dx*tanh(dx/2gamma) per axis), Q_ij the squared Euclidean distance.

The net-level factor [1 + beta (t1 w_tns(e) + t2 w_wns(e))] WL_e is applied here too, as its excess over the plain
wirelength: sum_e x_e WL_e with x_e = base_e * beta (t1 w_tns + t2 w_wns) (state['net_extra']). Scaling d.n_weight in
place does not reach the placer: Attar 1.0.4's global_place builds an Evaluator at its start and the Evaluator copies
the weights (wmask = torch.where(deg >= 2, d.n_weight, 0)), so the GP never sees later changes.

The term is evaluated outside global_place's CUDA graph and gets the same preconditioner as the wirelength's gradient
(global_place divides it), so nothing else has to be scaled here.
"""
import torch

from attar import Evaluator
from attar import metrics as M

from . import common


def make_extra(d, store, state, t1, t2, lin=True, quad=True, lin_scale=1.0, quad_scale=1.0):
    p_cell, size, off = d.p_cell, d.size, d.pin_off
    ev = Evaluator(d)
    cache = {}

    def view():
        """The pair list as pin-index views, cached until the weights change (the store's version)."""
        v = cache.get('v')
        if v is not None and v[0] == store.version:
            return v[1:]
        i, j, w1, w2 = store.pairs(t1, t2, lin, quad)
        if len(i) == 0:
            out = (i.new_zeros(0), i.new_zeros(0), i.new_zeros(0), w1, w2)
        else:
            sel, inv = torch.unique(torch.cat([i, j]), return_inverse=True)
            out = (sel, inv[:len(i)], inv[len(i):], w1, w2)
        cache['v'] = (store.version, *out)
        return out

    def extra(pos):
        val = 0.0
        grad = torch.zeros_like(pos)
        x = state.get('net_extra')
        if x is not None:
            nv = cache.get('net')
            if nv is None or nv[0] is not x:                    # only the pins of the weighted nets (memory)
                pm = torch.nonzero(x[ev.net] > 0).flatten()
                nv = (x, pm, ev.net[pm], p_cell[pm])
                cache['net'] = nv
            _x, pm, net_s, cell_s = nv
            if len(pm):
                g = common.gamma_of(state['ov'], state['gamma_base'])
                v_n, gp = M.wa_value_grad(d.pin_pos(pos)[pm], net_s, ev.E, g, x)
                grad.index_add_(0, cell_s, gp)
                val += float(v_n)
        if store.empty() or (not lin and not quad):
            return val, grad
        sel, ii, jj, w1, w2 = view()
        if len(sel) == 0:
            return val, grad
        pos = pos.detach().requires_grad_(True)
        p = pos[p_cell[sel]] + size[p_cell[sel]] * 0.5 + off[sel]
        dx = p[ii, 0] - p[jj, 0]
        dy = p[ii, 1] - p[jj, 1]
        v = p.new_zeros(())
        if lin:
            g = common.gamma_of(state['ov'], state['gamma_base'])
            v = v + lin_scale * (w1 * (dx * torch.tanh(dx / (2 * g)) + dy * torch.tanh(dy / (2 * g)))).sum()
        if quad:
            v = v + quad_scale * (w2 * (dx * dx + dy * dy)).sum()
        v = v * state['beta']
        gpp, = torch.autograd.grad(v, pos)
        return val + float(v.detach()), grad + gpp

    return extra


def value_at(d, store, pos, t1, t2, lin=True, quad=True, ov=0.0, gamma_base=None, lin_scale=1.0,
             quad_scale=1.0):
    """The pin-pair terms' value and gradient on every cell at beta = 1 (diagnostics and the beta calibration)."""
    p_cell, size, off = d.p_cell, d.size, d.pin_off
    i, j, w1, w2 = store.pairs(t1, t2, lin, quad)
    grad = torch.zeros_like(pos)
    if len(i) == 0 or (not lin and not quad):
        return 0.0, grad
    pos = pos.detach().requires_grad_(True)
    p = pos[p_cell] + size[p_cell] * 0.5 + off
    dx = p[i, 0] - p[j, 0]
    dy = p[i, 1] - p[j, 1]
    val = p.new_zeros(())
    if lin:
        g = common.gamma_of(ov, gamma_base or common.gamma_base(d))
        val = val + lin_scale * (w1 * (dx * torch.tanh(dx / (2 * g)) + dy * torch.tanh(dy / (2 * g)))).sum()
    if quad:
        val = val + quad_scale * (w2 * (dx * dx + dy * dy)).sum()
    grad, = torch.autograd.grad(val, pos)
    return float(val.detach()), grad.detach()

In [ ]:
%%writefile src/weights.py
"""The six cumulative weights (paper eqs. 17, 18 and section 3.4).

Every m iterations, from iteration `start`, the placement is timed and every violating endpoint's worst path is
walked. Each wire step (driver pin i -> sink pin j, net e) adds, with r / c the unit wire RC and C_lj the sink pin's
capacitance:

  net   w_tns(e)      += b_i*c + ((1 + a_j*K_s) * R_i + a_j*v_i) * c          (all paths)
        w_wns(e)      += the same                                            (the worst path only)
  lin   w_tns(L_ij)   += (1 + a_j*K_s) * r * C_lj                            (all paths)
  quad  w_tns(L^2_ij) += (1 + a_j*K_s) * r * c / 2                           (all paths)

with a_j the cell-delay sensitivity to the input slew of the arc leaving pin j, b_i / R_i the load sensitivity and
driver resistance of the arc arriving at the driver pin i, and v_i its output-slew sensitivity to the load. F_j (the
number of critical paths through pin j) needs no separate pass: a path through pin j visits it once, so summing over
paths counts it. Weights are cumulative across updates: a pair that was critical once keeps being pulled.

The pin-pair weights are kept in a sorted key array (key = i * P + j), merged with each update's pairs.
"""
import time

import torch

from . import coeffs


class Weights:
    """scheme='derived': the paper's six cumulative weights (ps/um, ps/um^2).
    scheme='date25': DATE25-TDP's pin-pair scheme (net_weighting_scheme.h, NetWeighting<PIN2PIN>): a pair starts at
    pin2pin_min_weight (10), each further critical path adds 0.2 * slack/wns, capped at pin2pin_max_weight (50)."""

    def __init__(self, d, dev=None, scheme='derived'):
        self.d = d
        self.dev = dev or d.device
        self.E = len(d.n_weight)
        self.P = int(d.p_cell.numel())
        self.net_tns = torch.zeros(self.E, device=self.dev)
        self.net_wns = torch.zeros(self.E, device=self.dev)
        self.net_tns_nob = torch.zeros(self.E, device=self.dev)      # A4: without the b_i*c term
        self.keys = torch.zeros(0, dtype=torch.long, device=self.dev)
        self.lin_tns = self.lin_wns = self.quad_tns = self.quad_wns = torch.zeros(0, device=self.dev)
        self.ks = self.r = self.c = None
        self.updates = 0
        self.version = 0
        self.scheme = scheme
        self.min_weight = 10.0          # pin2pin_min_weight
        self.max_weight = 50.0          # pin2pin_max_weight
        self.accumulate = 0.2           # pin2pin_accumulate_weight
        self.stats = {}

    # -- filling ----------------------------------------------------------------------------------------------------
    def update(self, res, cp=None):
        """Accumulate one round of weights from a TimingResult (worst paths of the violating endpoints). cp: another
        path set in CriticalPaths form (e.g. OpenTimer's paths, src/p11_dreamplace.py); the coefficients still come
        from res's operating point."""
        t0 = time.time()
        T = res.timer
        d = self.d
        dev = self.dev
        self.ks, self.r, self.c = coeffs.ks(T), float(T.r_sig), float(T.c_sig)
        cp = res.critical_paths() if cp is None else cp
        n_paths, R = len(cp), len(cp.pin)
        st = {'paths': n_paths, 'rows': R, 'ks': self.ks, 'r': self.r, 'c': self.c}
        if n_paths == 0 or R < 2:
            st['skipped'] = True
            self.stats = st
            return st

        ptr, kk_all = cp.ptr, cp.path_of()
        pin, rf, wire = cp.pin, cp.rf, cp.wire
        same = (kk_all[1:] == kk_all[:-1]) & wire[1:]              # wire steps only (driver i -> sink j); a cell arc
        idx = torch.arange(1, R, device=dev)[same]                 # (input -> output) is no pin pair. Row of pin j
        kk = kk_all[idx]
        has_prev = (idx - 1) > ptr[kk]
        has_next = (idx + 1) < ptr[kk + 1]
        prev_cell = has_prev & (~wire[idx - 1])
        next_cell = has_next & (~wire[(idx + 1).clamp(max=R - 1)])

        i, j = pin[idx - 1], pin[idx]
        rfi, rfj = rf[idx - 1], rf[idx]
        i_prev = pin[(idx - 2).clamp(min=0)]
        u_next = pin[(idx + 1).clamp(max=R - 1)]
        rf_prev, rf_next = rf[(idx - 2).clamp(min=0)], rf[(idx + 1).clamp(max=R - 1)]

        prev_arc = coeffs.arc_index(T, i_prev, i)
        next_arc = coeffs.arc_index(T, j, u_next)
        a_p, b_p, u_p, v_p = coeffs.sens(T, torch.where(prev_cell, prev_arc, -1), rfi, rf_prev)
        a_n, b_n, u_n, v_n = coeffs.sens(T, torch.where(next_cell, next_arc, -1), rf_next, rfj)

        a_j, b_i, v_i, R_i = a_n, b_p, v_p, b_p
        r, c, ks = self.r, self.c, self.ks
        C_lj = T.pin_cap[j, rfj]
        one_plus = 1.0 + a_j * ks

        supply = (one_plus * R_i + a_j * v_i) * c
        w_net = b_i * c + supply
        w_net_nob = supply
        w_lin = one_plus * r * C_lj
        w_quad = one_plus * r * c / 2.0

        worst = kk == 0
        net = d.p_net[i]
        zero = torch.zeros((), device=dev)
        self.net_tns.index_add_(0, net, w_net)
        self.net_tns_nob.index_add_(0, net, w_net_nob)
        self.net_wns.index_add_(0, net, torch.where(worst, w_net, zero))

        if self.scheme == 'date25':
            new = self._merge_date25(i * self.P + j, kk, cp)
        else:
            new = self._merge(i * self.P + j, w_lin, torch.where(worst, w_lin, zero), w_quad,
                              torch.where(worst, w_quad, zero))

        self.updates += 1
        st.update(edges=int(len(i)), new_pairs=int(new), pairs=int(len(self.keys)),
                  f_max=int(cp.pin_count().max().item()) if R else 0,
                  w_net_max=float(self.net_tns.max().item()), w_net_mean=float(self.net_tns.mean().item()),
                  n_net=int((self.net_tns > 0).sum().item()),
                  seconds=round(time.time() - t0, 2))
        self.stats = st
        return st

    def _merge_date25(self, key, kk, cp):
        """DATE25-TDP's pin-pair weights (net_weighting_scheme.h, PIN2PIN): a new pair starts at 10, every further
        critical path adds 0.2 * slack/wns (1 at the worst path), capped at 50. Stored in quad_tns."""
        dev = self.dev
        wns = float(cp.slack[0]) if len(cp.slack) else -1.0
        wns = wns if wns < 0 else -1.0
        ratio = (cp.slack[kk] / wns).clamp(0.0, 1.0)
        inc = self.accumulate * ratio
        uk, inv = torch.unique(key, return_inverse=True)
        tot = torch.zeros(len(uk), device=dev).index_add_(0, inv, inc)
        if len(self.keys) == 0:
            self.keys = uk
            self.quad_tns = torch.full((len(uk),), self.min_weight, device=dev)
            self.lin_tns = torch.zeros(len(uk), device=dev)
            self.lin_wns = torch.zeros(len(uk), device=dev)
            self.quad_wns = torch.zeros(len(uk), device=dev)
            return len(uk)
        merged = torch.unique(torch.cat([self.keys, uk]))
        old_pos = torch.searchsorted(merged, self.keys)
        new_pos = torch.searchsorted(merged, uk)
        q = torch.zeros(len(merged), device=dev).index_add_(0, old_pos, self.quad_tns)
        hit = torch.zeros(len(merged), dtype=torch.bool, device=dev)
        hit[old_pos] = True
        q.index_add_(0, new_pos, torch.where(hit[new_pos], tot, torch.zeros_like(tot)))
        q = torch.where(hit, q.clamp(max=self.max_weight), torch.full_like(q, self.min_weight))
        self.keys, self.quad_tns = merged, q
        self.lin_tns = torch.zeros(len(merged), device=dev)
        self.lin_wns = torch.zeros(len(merged), device=dev)
        self.quad_wns = torch.zeros(len(merged), device=dev)
        return len(uk)

    def _merge(self, key, lin_tns, lin_wns, quad_tns, quad_wns):
        """Merge this update's pin pairs into the sorted key array (cumulative)."""
        uk, inv = torch.unique(key, return_inverse=True)
        nf = torch.zeros(4, len(uk), device=self.dev)
        nf[0].index_add_(0, inv, lin_tns)
        nf[1].index_add_(0, inv, lin_wns)
        nf[2].index_add_(0, inv, quad_tns)
        nf[3].index_add_(0, inv, quad_wns)
        if len(self.keys) == 0:
            self.keys, self.lin_tns, self.lin_wns, self.quad_tns, self.quad_wns = uk, nf[0], nf[1], nf[2], nf[3]
            return len(uk)
        merged = torch.unique(torch.cat([self.keys, uk]))
        old = torch.searchsorted(merged, self.keys)
        new = torch.searchsorted(merged, uk)
        out = torch.zeros(4, len(merged), device=self.dev)
        for t, src in ((0, self.lin_tns), (1, self.lin_wns), (2, self.quad_tns), (3, self.quad_wns)):
            out[t].index_add_(0, old, src)
            out[t].index_add_(0, new, nf[t])
        self.keys, self.lin_tns, self.lin_wns, self.quad_tns, self.quad_wns = merged, out[0], out[1], out[2], out[3]
        self.version += 1
        return len(uk)

    # -- the objective's view ---------------------------------------------------------------------------------------
    def net_weight(self, base, beta, t1, t2, skip_b=False):
        """d.n_weight * (1 + beta (t1 w_tns(e) + t2 w_wns(e))) -- paper eq. 19's net-level factor."""
        w = self.net_tns_nob if skip_b else self.net_tns
        return base * (1.0 + beta * (t1 * w + t2 * self.net_wns))

    def pairs(self, t1, t2, lin=True, quad=True):
        """(i, j, w1, w2): the pin pairs and their linear / quadratic weights * t1 / t2, as eqs. 20-21 use them."""
        if len(self.keys) == 0:
            z = self.keys
            return z, z, self.lin_tns, self.lin_tns
        i = torch.div(self.keys, self.P, rounding_mode='floor')
        j = self.keys - i * self.P
        w1 = (t1 * self.lin_tns + t2 * self.lin_wns) if lin else None
        w2 = (t1 * self.quad_tns + t2 * self.quad_wns) if quad else None
        return i, j, w1, w2

    def empty(self):
        return len(self.keys) == 0 and float(self.net_tns.abs().sum()) == 0.0

In [ ]:
%%writefile src/coeffs.py
"""Linearization coefficients at the current timing working point (paper eqs. 11-16).

Everything the linear model needs is already in Attar's timer after a run (`res.timer`):

  c_fr / c_to / c_lib / c_sense_lib / c_type   the delay arcs (from pin, to pin, NLDM function, sense, type)
  t_ix1 / t_ix2 / t_val / t_n                  the NLDM tables, slot orf = delay, slot 2 + orf = output slew
  smax / load                                  input slew at a pin, total load at a driver pin (fF)
  pin_cap                                      a pin's capacitance (fF)
  r_sig / c_sig                                wire resistance (kOhm/um) and capacitance (fF/um)
  t_thr[lib, orf]                              (vth, slew_lower, slew_upper, derate)

The coefficients are the table's derivatives at the working point (not a global fit): a = dD/dslew_in, b = dD/dload,
u = dSlew/dslew_in, v = dSlew/dload. b is at once the linear delay model's load coefficient and the driver resistance
(OpenSTA's gateModelRd is the same table slope), so R_i = b_i in the paper's eq. 17.
"""
import math

import torch

from attar.timing.sta import nldm

_CACHE = '_p11_coeffs'


def ks(T):
    """K_s: the slew-to-Elmore constant of the library's slew thresholds, ln(slew_upper / slew_lower).

    The paper calls it "a constant defined in the timing library". ICCAD 2015's libraries measure slew 20 %-80 %
    (K_s = ln 4 = 1.386); ASAP7's 10 %-90 % (ln 9 = 2.197). Read from the tables, never hard-coded."""
    cache = getattr(T, _CACHE, None)
    if cache is not None and 'ks' in cache:
        return cache['ks']
    thr = getattr(T, 't_thr', None)
    ratio = 4.0
    if thr is not None and thr.numel():
        vl, vh = thr[..., 1], thr[..., 2]
        ok = (vh > vl) & (vl > 0) & torch.isfinite(vh)
        if bool(ok.any()):
            ratio = float((vh[ok] / vl[ok]).median().item())
    val = math.log(ratio) if ratio > 1 else math.log(4.0)
    _cache(T)['ks'] = val
    return val


def _cache(T):
    c = getattr(T, _CACHE, None)
    if c is None:
        c = {}
        setattr(T, _CACHE, c)
    return c


def arc_index(T, fr, to):
    """The timer's delay arc index for each (from pin, to pin) pair, -1 where there is none.

    Several arcs can share a pair (a cell with two timing arcs into the same pin); the first by arc index wins, which
    is the one the levelized run uses for the arrival."""
    c = _cache(T)
    if 'keys' not in c:
        key = T.c_fr * T.P + T.c_to
        order = torch.argsort(key)
        c['keys'], c['order'] = key[order], order
    skey, order = c['keys'], c['order']
    q = fr * T.P + to
    pos = torch.searchsorted(skey, q).clamp(max=len(skey) - 1)
    idx = order[pos]
    return torch.where(skey[pos] == q, idx, torch.full_like(idx, -1))


def sens(T, arcs, orf, irf):
    """(a, b, u, v) per arc at the given output / input transitions: dD/dslew, dD/dload, dS/dslew, dS/dload.

    ps/ps, ps/fF, ps/ps, ps/fF. arcs < 0 (no such arc) gives zeros. Computed on the unique (arc, orf, irf) keys, so a
    critical-path's repeated arcs cost nothing extra."""
    dev = arcs.device
    live = arcs >= 0
    a0 = torch.where(live, arcs, torch.zeros_like(arcs))
    orf = orf.clamp(0, 1)
    irf = irf.clamp(0, 1)
    key = a0 * 4 + orf * 2 + irf
    uk, inv = torch.unique(key, return_inverse=True)
    ua, ur, ui = uk // 4, (uk % 4) // 2, uk % 2
    lib = T.c_lib[ua]
    x = T.smax[T.c_fr[ua], ui]
    y = T.load[T.c_to[ua], ur]
    x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).clamp(min=0.0).clone().requires_grad_(True)
    y = torch.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0).clamp(min=0.0).clone().requires_grad_(True)

    def look(slot):
        return nldm(T.t_ix1[lib, slot], T.t_ix2[lib, slot], T.t_val[lib, slot], T.t_n[lib, slot], x, y)

    D, S = look(ur), look(2 + ur)
    (ax, by), (ux, vy) = (torch.autograd.grad(D.sum(), (x, y), retain_graph=True),
                          torch.autograd.grad(S.sum(), (x, y)))
    zero = torch.zeros((), device=dev)
    out = []
    for t in (ax, by, ux, vy):
        t = torch.nan_to_num(t, nan=0.0, posinf=0.0, neginf=0.0)
        out.append(torch.where(live, t[inv], zero))
    return out


def slew_in(T, pin, rf):
    """The input slew at a pin (ps), for the probes."""
    return T.smax[pin, rf.clamp(0, 1)]

In [ ]:
%%writefile src/abacus.py
"""Abacus legalization (the paper's and DATE25-TDP's legalizer) for Attar designs of single-row-height cells.

  from src import abacus; stats = abacus.legalize(d)      # in place: d.pos of the movable cells

The algorithm is in abacus.cpp (built on first use with g++, called through ctypes). Fixed objects other than IO
terminals cut the rows into segments. Cells that find no room (none so far) fall back to Attar's incremental
legalizer. Checked with Attar's own Evaluator.legality().
"""
import ctypes
import os
import subprocess
import time

import numpy as np
import torch

from attar import Evaluator
from attar.io.distill import ASIDE, MACRO, TERMINAL

HERE = os.path.dirname(os.path.abspath(__file__))
SRC = os.path.join(HERE, 'abacus.cpp')
LIB = os.path.join(HERE, 'build', '_abacus.so')
_lib = None


def _load():
    global _lib
    if _lib is not None:
        return _lib
    if not os.path.exists(LIB) or os.path.getmtime(LIB) < os.path.getmtime(SRC):
        os.makedirs(os.path.dirname(LIB), exist_ok=True)
        tmp = LIB + '.%d' % os.getpid()
        subprocess.check_call(['g++', '-O3', '-std=c++17', '-shared', '-fPIC', '-o', tmp, SRC])
        os.replace(tmp, LIB)
    lib = ctypes.CDLL(LIB)
    D, I = ctypes.POINTER(ctypes.c_double), ctypes.POINTER(ctypes.c_int)
    lib.abacus.argtypes = [ctypes.c_int, D, D, D, ctypes.c_int, I, D, D, ctypes.c_int, D, D, D, D,
                           ctypes.c_double, ctypes.c_double, D, I]
    lib.abacus.restype = ctypes.c_int
    _lib = lib
    return lib


def _p(a, t):
    return a.ctypes.data_as(ctypes.POINTER(t))


def legalize(d, verbose=False):
    t0 = time.time()
    lib = _load()
    if d.row_box is None:
        raise ValueError('no rows')
    mv = d.movable.cpu()
    flags = d.flags.cpu()
    size = d.size.detach().cpu().double().numpy()
    pos = d.pos.detach().cpu().double().numpy()
    rb = d.row_box.detach().cpu().double().numpy()
    hb = float(d.row_h.min())
    if bool(((d.size[d.movable, 1] / hb).round() > 1).any()):
        raise NotImplementedError('multi-row cells: use naive.legalize')
    sw = float(d.row_site_w.float().median())
    order = np.argsort(rb[:, 1], kind='stable')
    rb = rb[order]
    ry = np.ascontiguousarray(rb[:, 1])
    ox0 = float(rb[:, 0].min())
    rx0 = np.ascontiguousarray(np.round((rb[:, 0] - ox0) / sw))
    rx1 = np.ascontiguousarray(np.floor((rb[:, 2] - ox0) / sw + 1e-6))
    R = len(ry)

    # fixed objects (not IO terminals) block the rows they overlap
    fx = torch.nonzero(~mv & ((flags & (TERMINAL | ASIDE)) == 0)).flatten().numpy()
    x0, y0 = pos[fx, 0], pos[fx, 1]
    x1, y1 = x0 + size[fx, 0], y0 + size[fx, 1]
    eps = 1e-6
    r_lo = np.searchsorted(ry + hb, y0 + eps, side='right')        # first row whose top is above y0
    r_hi = np.searchsorted(ry, y1 - eps, side='left') - 1          # last row whose bottom is below y1
    n = np.clip(r_hi - r_lo + 1, 0, None)
    keep = n > 0
    krow = (np.repeat(r_lo[keep], n[keep]) + np.concatenate([np.arange(k) for k in n[keep]])
            if keep.any() else np.zeros(0, dtype=np.int64))
    kx0 = np.repeat(np.floor((x0[keep] - ox0) / sw + eps), n[keep])
    kx1 = np.repeat(np.ceil((x1[keep] - ox0) / sw - eps), n[keep])
    krow = np.ascontiguousarray(krow, dtype=np.int32)
    kx0, kx1 = np.ascontiguousarray(kx0), np.ascontiguousarray(kx1)

    # movable cells: target x in sites, y in um, widths in sites, weights = pin counts (DREAMPlace's node weights)
    mi = torch.nonzero(mv).flatten().numpy()
    cx = np.ascontiguousarray((pos[mi, 0] - ox0) / sw)
    cy = np.ascontiguousarray(pos[mi, 1])
    cw = np.ascontiguousarray(np.round(size[mi, 0] / sw))
    pins = torch.bincount(d.p_cell.cpu(), minlength=len(flags)).double().numpy()
    ce = np.ascontiguousarray(np.maximum(pins[mi], 1.0))
    N = len(mi)
    ox = np.zeros(N)
    orow = np.zeros(N, dtype=np.int32)
    D, I = ctypes.c_double, ctypes.c_int
    failed = lib.abacus(R, _p(ry, D), _p(rx0, D), _p(rx1, D), len(krow), _p(krow, I), _p(kx0, D), _p(kx1, D),
                        N, _p(cx, D), _p(cy, D), _p(cw, D), _p(ce, D), hb, sw, _p(ox, D), _p(orow, I))
    ok = orow >= 0
    new = pos.copy()
    new[mi[ok], 0] = ox0 + ox[ok] * sw
    new[mi[ok], 1] = ry[orow[ok]]
    p = torch.as_tensor(new, dtype=d.pos.dtype, device=d.device)
    disp = np.abs(new[mi] - pos[mi]).sum(1)
    d.pos = p
    stats = {'mode': 'abacus', 'failed': int(failed), 'seconds': round(time.time() - t0, 2),
             'disp_mean_um': float(disp.mean()), 'disp_max_um': float(disp.max()), 'obstacle_intervals': int(len(krow))}
    if failed:
        from attar import naive
        cells = torch.zeros(len(flags), dtype=torch.bool)
        cells[torch.as_tensor(mi[~ok])] = True
        naive.legalize(d, cells=cells.to(d.device), verbose=False)
    if verbose:
        print('  abacus:', stats)
    return stats

In [ ]:
%%writefile src/abacus.cpp
// Abacus legalization (Spindler, Schlichtmann, Johannes, ISPD 2008) for single-row-height standard cells, the
// legalizer DATE25-TDP and the paper run after global placement (DREAMPlace's abacus_legalize).
//
// Units: x in sites from the rows' origin (integers for segment bounds and widths), rows by index. Fixed objects cut
// the rows into segments. Cells are taken in order of their global-placement x; for each, rows are tried outward
// from the nearest one while the vertical displacement alone is below the best cost so far; in a row, the segment
// that holds the cell's x and the nearest segments with room on either side are tried. A trial appends the cell to
// the segment's last cluster and collapses (PlaceRow) on a copy of the tail; the cost is the cell's displacement
// (|dx| + |dy|, um). The final cluster positions are rounded to the site grid (exact: widths are integers, so the
// order and the abutments survive rounding).
//
// Build: g++ -O3 -shared -fPIC -o _abacus.so abacus.cpp     (called through ctypes, src/abacus.py)
#include <algorithm>
#include <cmath>
#include <cstdint>
#include <vector>

namespace {

struct Cluster {
  double e, q, w, x;  // total weight, sum e_i (x_i - offset_i), width, position
  int first, last;    // cells (indices into the segment's cell list)
};

struct Segment {
  int row;
  double x0, x1;  // sites
  double used = 0;
  std::vector<int> cells;  // in placement order
  std::vector<double> off; // each cell's offset inside its cluster at insertion (recomputed at the end)
  std::vector<Cluster> cl;
};

inline double clampd(double v, double lo, double hi) { return v < lo ? lo : (v > hi ? hi : v); }

// position of a new cell (target x, width w, weight e) appended to seg; no modification
double trial(const Segment& s, double x, double w, double e) {
  double ce = e, cq = e * x, cw = w;
  double cx = clampd(x, s.x0, s.x1 - cw);
  double offset = 0.0;  // the new cell's offset in the merged cluster
  int k = (int)s.cl.size() - 1;
  while (k >= 0) {
    const Cluster& p = s.cl[k];
    if (p.x + p.w <= cx + 1e-9) break;
    // merge p (left) with the current cluster (right): q_new = q_p + q_c - e_c * w_p
    cq = p.q + cq - ce * p.w;
    ce = p.e + ce;
    offset += p.w;
    cw = p.w + cw;
    cx = clampd(cq / ce, s.x0, s.x1 - cw);
    --k;
  }
  return cx + offset;
}

void insert(Segment& s, int cell, double x, double w, double e) {
  int idx = (int)s.cells.size();
  s.cells.push_back(cell);
  s.off.push_back(0.0);
  s.used += w;
  Cluster c{e, e * x, w, clampd(x, s.x0, s.x1 - w), idx, idx};
  while (!s.cl.empty()) {
    Cluster& p = s.cl.back();
    if (p.x + p.w <= c.x + 1e-9) break;
    Cluster m{p.e + c.e, p.q + c.q - c.e * p.w, p.w + c.w, 0.0, p.first, c.last};
    m.x = clampd(m.q / m.e, s.x0, s.x1 - m.w);
    s.cl.pop_back();
    c = m;
  }
  s.cl.push_back(c);
}

}  // namespace

extern "C" {

// rows: R rows, row r at y = ry[r] (um), spanning sites [rx0[r], rx1[r]) (sites from the origin).
// obstacles: K blocked intervals already in (row, site x0, site x1) form (x0 floored, x1 ceiled by the caller).
// cells: N cells with target (cx sites, cy um), width cw (sites, integer), weight ce.
// out: ox (sites, integer-valued), orow (row index). Returns the number of cells that found no room (placed: -1).
int abacus(int R, const double* ry, const double* rx0, const double* rx1, int K, const int* krow, const double* kx0,
           const double* kx1, int N, const double* cx, const double* cy, const double* cw, const double* ce,
           double row_h, double site_w, double* ox, int* orow) {
  // segments per row
  std::vector<std::vector<std::pair<double, double>>> blk(R);
  for (int k = 0; k < K; ++k)
    if (krow[k] >= 0 && krow[k] < R) blk[krow[k]].push_back({kx0[k], kx1[k]});
  std::vector<Segment> seg;
  std::vector<int> row_first(R + 1, 0);
  for (int r = 0; r < R; ++r) {
    row_first[r] = (int)seg.size();
    auto& b = blk[r];
    std::sort(b.begin(), b.end());
    double x = rx0[r];
    for (auto& iv : b) {
      double a = std::max(iv.first, rx0[r]), z = std::min(iv.second, rx1[r]);
      if (z <= a) continue;
      if (a > x) { Segment s; s.row = r; s.x0 = x; s.x1 = a; seg.push_back(std::move(s)); }
      x = std::max(x, z);
    }
    if (rx1[r] > x) { Segment s; s.row = r; s.x0 = x; s.x1 = rx1[r]; seg.push_back(std::move(s)); }
  }
  row_first[R] = (int)seg.size();

  std::vector<int> order(N);
  for (int i = 0; i < N; ++i) order[i] = i;
  std::sort(order.begin(), order.end(), [&](int a, int b) { return cx[a] < cx[b]; });

  int failed = 0;
  for (int i : order) {
    double x = cx[i], y = cy[i], w = cw[i], e = ce[i];
    int r0 = (int)std::lround((y - ry[0]) / row_h);
    r0 = std::max(0, std::min(R - 1, r0));
    double best = 1e300;
    int best_seg = -1;
    for (int d = 0; d < R; ++d) {
      bool any = false;
      for (int sgn = 0; sgn < 2; ++sgn) {
        if (d == 0 && sgn == 1) continue;
        int r = sgn ? r0 - d : r0 + d;
        if (r < 0 || r >= R) continue;
        any = true;
        double dy = std::fabs(ry[r] - y);
        if (dy >= best) continue;
        int a = row_first[r], b = row_first[r + 1];
        if (a == b) continue;
        // the segment holding x (or the first to its right), then the nearest with room on either side
        int h = (int)(std::upper_bound(seg.begin() + a, seg.begin() + b, x,
                                       [](double v, const Segment& s) { return v < s.x1; }) - seg.begin());
        int cand[3] = {-1, -1, -1};
        if (h < b && seg[h].x0 <= x && seg[h].x1 - seg[h].x0 - seg[h].used >= w) cand[0] = h;
        for (int k = std::min(h, b - 1); k >= a; --k)
          if (k != cand[0] && seg[k].x1 - seg[k].x0 - seg[k].used >= w && seg[k].x1 <= x + w) { cand[1] = k; break; }
        for (int k = h; k < b; ++k)
          if (k != cand[0] && seg[k].x1 - seg[k].x0 - seg[k].used >= w && seg[k].x0 >= x - 1e-9) { cand[2] = k; break; }
        if (cand[0] < 0 && cand[1] < 0 && cand[2] < 0) {      // nothing near: any segment with room
          for (int k = a; k < b; ++k)
            if (seg[k].x1 - seg[k].x0 - seg[k].used >= w) { cand[1] = k; break; }
        }
        for (int c : cand) {
          if (c < 0) continue;
          double nx = trial(seg[c], x, w, e);
          double cost = std::fabs(nx - x) * site_w + dy;
          if (cost < best) { best = cost; best_seg = c; }
        }
      }
      if (!any) break;
      if (d * row_h >= best) break;
    }
    if (best_seg < 0) { ox[i] = x; orow[i] = -1; ++failed; continue; }
    insert(seg[best_seg], i, x, w, e);
  }

  // final positions: clusters rounded to the site grid, cells packed inside
  for (auto& s : seg) {
    for (auto& c : s.cl) {
      double px = std::round(c.x);
      if (px < s.x0) px = std::ceil(s.x0);
      if (px + c.w > s.x1) px = std::floor(s.x1 - c.w);
      for (int k = c.first; k <= c.last; ++k) {
        int cell = s.cells[k];
        ox[cell] = px;
        orow[cell] = s.row;
        px += cw[cell];
      }
    }
  }
  return failed;
}
}

In [ ]:
%%writefile src/relegalize.py
"""Legalize every saved global placement (gp_pos.pt) with Abacus and measure it: metrics_abacus.json next to the
run's metrics.json (which holds the Attar-legalizer numbers).

  python -m src.relegalize --root results_v2/suite [--force]
"""
import argparse
import glob
import json
import os
import time

import torch

from . import abacus, common


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--root', default='results_v2/suite')
    ap.add_argument('--force', action='store_true')
    ap.add_argument('--device', default='cpu')
    a = ap.parse_args()
    runs = sorted(glob.glob(os.path.join(a.root, '*', '*', 'seed*', 'gp_pos.pt')))
    loaded = {}
    for f in runs:
        rd = os.path.dirname(f)
        out = os.path.join(rd, 'metrics_abacus.json')
        if os.path.exists(out) and not a.force:
            continue
        method, design, seed = rd.split(os.sep)[-3:]
        if design not in loaded:
            loaded.clear()
            loaded[design] = common.load(design, dev=a.device, unplace=False)
        d = loaded[design]
        gp = torch.load(f).to(d.device)
        d.pos = gp
        t0 = time.time()
        st = abacus.legalize(d)
        m = common.measure(d)
        m.update(method=method, design=design, seed=int(seed[4:]), lg=st, lg_seconds=round(time.time() - t0, 1))
        json.dump(m, open(out, 'w'), indent=1)
        torch.save(d.pos.detach().cpu(), os.path.join(rd, 'pos_abacus.pt'))
        print('%-5s %-12s %s  legal %s  HPWL %.4e  WNS %.0f  TNS %.4e  disp %.1f um' % (
            method, design, seed, m['legal'], m['hpwl_um'], m['wns'], m['tns'], st['disp_mean_um']), flush=True)


if __name__ == '__main__':
    main()

## 1. DATE25 global placement
第一次執行會先下載 superblue18 的設計檔。

In [ ]:
!python -m src.run --methods D25 --seeds 0 --designs superblue18 --start-ov 0.55 --bins 512 --quiet --gp tdp --beta 2.5e-5 --beta-units site --m-d25 15 --device cuda --root results

## 2. Abacus 合法化，再量 timing

In [ ]:
!python -m src.relegalize --root results --device cuda

## 3. 結果

In [ ]:
import json
SEED = 0
m = json.load(open(f'results/D25/superblue18/seed{SEED}/metrics_abacus.json'))
print('DATE25  seed %d：TNS %.4g ps  WNS %.1f ps  HPWL %.4g um' % (SEED, m['tns'], m['wns'], m['hpwl_um']))
print('參考（我們的執行）：seed 0 TNS -5.327e+06 / WNS -8548.2；3 個 seed 平均 TNS -5.415e+06 / WNS -8922.4')

同一個 seed 每次執行結果也會不同（GPU 浮點運算順序不同）。我們在 superblue18 seed 0 各跑了四次：P11 的 TNS 很穩定（-3.38e6 到 -3.51e6），對正常的 DATE25 改善 1.52–1.59 倍；WNS 只由一條最差路徑決定，單次在 0.91–1.08 倍之間浮動，要看多個 seed 的平均（3 個 seed 平均 1.11 倍）。DATE25 對照組的波動更大：同樣 seed 0 跑四次，三次 TNS 約 -5.3e6 到 -5.5e6，有一次第一次權重更新後就走偏，停在 -8.3e6。單次結果請當作參考，多跑幾次或換 seed 比較可靠。要換 seed，把第 1 節的 `--seeds 0` 和第 3 節的 `SEED = 0` 一起改掉。